# <span style="color:blue">Project X4 v7.0 Ship rating. Part 1 </span>

## Introduction <a class="anchor" id="id_0"></a> 

### Table of contents 

* [Introduction](#id_0)  
  * [Libraries](#id_1)
  * [Project description](#id_1a)
  * [File folders](#id_1b)
  * [Custom functions](#id_1c)
* [Acquaring the data](#id_2)
  * [Creating the list of needed files](#id_2a)  
  * [Creating the list of files with ship configurations](#id_2b_1)
  * [Creating the list of ship names](#id_2c)
  * [Creating the table with ship stats](#id_2d)
    * [Creation](#id_2d_1)
    * [Clearing the table](#id_2d_2) 
* [Get the data about ship equipment](#id_3) 
  * [How find data about the number of equipment slots](#id_2f)
  * [Shields](#id_3a) 
  * [Engines](#id_3b)
  * [The number of S/M docks](#id_3c)
* [Saving the tables](#id_4)

### Libraries <a class="anchor" id="id_1"></a> 

In [1]:
import pandas as pd
import numpy as np 
import os
import re
from bs4 import BeautifulSoup
import lxml
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)
#pd.set_option('max_colwidth', 300)

### Description <a class="anchor" id="id_1a"></a> 

X4: Foundations - space sandbox game, where a player can build factories, trade, wage ware, board ships and complete quests in an evolving world with beatiful graphic and visual effects. The game was released by the company Egosoft in 2018 and byhte moment has 5 dlcs. 
This project is done for my favorite game in the name of curiosity. The aim of it is to prove my skills at data extraction and to see how ship's base parameters are distributed between many in-game spaceships in different categories, rank them and draw some diagrams to see the big picture.
The ranking is done only for ships a player can fly on.  
The information is extracted for all ships in the game.  
The information about shields is extracted for all shields in the game.  
The information about engines is extracted for all ship engines, except engines for mines, drones, missiles, spacesuits.  

The part 1 is dedicated to extracting needed data from the game files, formatting and storing into dataframes.  
The part 2 is dedicated to calculating of additional parameters, constructing the ranking system and drawing diagrams.

To show the ranks of ships I constructed dashboard on th  public Tableau server and you can see it on the link:
https://public.tableau.com/views/TheshipratingforthegameX4Foundations/Shiprating?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link

### File folders <a class="anchor" id="id_1b"></a> 

In [2]:
#Folders with game objects
units_folder = r'H:\Steam\X4\Unpacked\assets\units' # каталог с файлами xml, которых сохранена информация о игровых объектах.
boron_dlc_folder = r'H:\Steam\X4\Unpacked\extensions\boron_dlc\assets\units' # The folder with game objects from dlc "Kingdom End"
avarice_dlc_folder = r'H:\Steam\X4\Unpacked\extensions\avarice_dlc\assets\units' # The folder with game objects from dlc "Tides of Avarice"
terran_dlc_folder = r'H:\Steam\X4\Unpacked\extensions\terran_dlc\assets\units'  # The folder with game objects from dlc "Cradle of humanity"
split_dlc_folder = r'H:\Steam\X4\Unpacked\extensions\split_dlc\assets\units' # The folder with game objects from dlc "Split Vendetta"
timelines_dlc_folder = r'H:\Steam\X4\Unpacked\extensions\timelines_dlc\assets\units' # The folder with game objects from dlc "Timelines"
# Folders with text informations (names, descriptions etc)
descriptions_eng = r'H:\Steam\X4\Unpacked\t\0001-l044.xml' #for english version
descriptions_ru = r'H:\Steam\X4\Unpacked\t\0001-l007.xml' #for  russian version
# A very problematic file without useful information but caused many errors
problemic_file = r'H:\Steam\X4\Unpacked\extensions\boron_dlc\assets\units\size_l\ship_bor_l_miner_solid_01_macro.xml'
# Folder to save extracted tables
save_folder = r'H:\Steam\X4\SavedTables' 

Shield files:

In [3]:
main_catalog = r'h:/Steam/X4/Unpacked/assets/props'
# Путь к каталогу с щитами
cat_shields = r'h:\Steam\X4\Unpacked\assets\props\SurfaceElements\macros'
#dlc ships shields 
split_shields = r'h:\Steam\X4\Unpacked\extensions\split_dlc\assets\props\surfaceelements\macros'
avarice_shields = r'h:\Steam\X4\Unpacked\extensions\avarice_dlc\assets\props\surfaceelements\macros'
terran_shields = r'h:\Steam\X4\Unpacked\extensions\terran_dlc\assets\props\surfaceelements\macros'
boron_shields = r'h:\Steam\X4\Unpacked\extensions\boron_dlc\assets\props\surfaceelements\macros'
timelines_shields = r'h:\Steam\X4\Unpacked\extensions\timelines_dlc\assets\props\surfaceelements\macros'

Engine files:

In [4]:
# Path to folder with engines and thrusters
cat_engines  = r'h:/Steam/X4/Unpacked/assets/props/Engines/macros'
cat_thrusters = r'h:/Steam/X4/Unpacked/assets/props/Engines/macros'
#dlc ships shields 
split_engines = r'h:\Steam\X4\Unpacked\extensions\split_dlc\assets\props\Engines\macros'
avarice_engines = r'h:\Steam\X4\Unpacked\extensions\avarice_dlc\assets\props\engines\macros'
terran_engines = r'h:\Steam\X4\Unpacked\extensions\terran_dlc\assets\props\engines\macros'
boron_engines = r'h:\Steam\X4\Unpacked\extensions\boron_dlc\assets\props\engines\macros'
timelines_engines = r'h:\Steam\X4\Unpacked\extensions\timelines_dlc\assets\props\engines\macros'

In [5]:
# Storage files
storage_files = r'H:\Steam\X4\Unpacked\assets\props\StorageModules\macros'
split_storage = r'H:\Steam\X4\Unpacked\extensions\split_dlc\assets\props\storagemodules\macros'
avarice_storage = r'H:\Steam\X4\Unpacked\extensions\avarice_dlc\assets\props\storagemodules\macros'
boron_storage = r'H:\Steam\X4\Unpacked\extensions\boron_dlc\assets\props\storagemodules\macros'
terran_storage = r'H:\Steam\X4\Unpacked\extensions\terran_dlc\assets\props\storagemodules\macros'
timelines_storage = r'H:\Steam\X4\Unpacked\extensions\timelines_dlc\assets\props\storagemodules\macros'

In [6]:
# folders with files containing data about the number of S/M docks and ship capacity
folder_docks = r'H:\Steam\X4\Unpacked\assets\structures\dock'
#dlc ships docks data
base_game_docks = r'h:\Steam\X4\Unpacked\assets\structures\dock\macros'
split_docks = r'h:\Steam\X4\Unpacked\extensions\split_dlc\assets\structures\dock\macros'
avarice_docks = r'h:\Steam\X4\Unpacked\extensions\avarice_dlc\assets\structures\dock\macros'
terran_docks = r'h:\Steam\X4\Unpacked\extensions\terran_dlc\assets\structures\dock\macros'
boron_docks = r'h:\Steam\X4\Unpacked\extensions\boron_dlc\assets\structures\dock\macros'
timelines_docks = r'h:\Steam\X4\Unpacked\extensions\timelines_dlc\assets\structures\dock\macros'

dock_folders = [base_game_docks, split_docks, avarice_docks, terran_docks, boron_docks, timelines_docks]

### Необходимые функции <a class="anchor" id="id_1c"></a> 

In [7]:
#The function to flatten lists
def flatten(l):
    return [item for sublist in l for item in sublist]

In [8]:
# Fuction to show missed data in a table
def missing_values_tab(df):
    #Count missed values
    mis_val = df.isnull().sum()
    #Count share of missed values of all data 
    mis_val_percent = round(100 * df.isnull().sum() / len(df),2)
    
    mv_table = pd.concat([mis_val, mis_val_percent], axis=1)
    
    mv_table = mv_table.rename(columns = {0 : 'missed_values', 1 : '%_of_all'})
    
    mv_table['data_type'] = df.dtypes
    mv_table = mv_table[mv_table.iloc[:,1]!=0].sort_values(by='missed_values',ascending=False)
    print ("Dataframe contains " + str(df.shape[1]) + " columns and " + str(df.shape[0]) + " strings.\n")
    print("It has  " + str(mv_table.shape[0]) +" columns with missed values")    
    return display(mv_table) 

In [9]:
def calculate_dock_class(data):
    """Function to calculate the class of ship bays and ship storages"""
    try:
        if re.match('\w+_(s)_\w+', data):
            return "S"
        if re.match('\w+_(m)_\w+', data):
            return "M"
        if re.match('\w+_(l)_\w+', data):
            return "L"
        if re.match('\w+_(xl)_\w+', data):
            return "XL"
        if re.match('\w+_(xs)_\w+', data):
            return "XS"
    except:
            return np.nan

## Acquaring the data <a class="anchor" id="id_2"></a> 

### Creating the list of needed files <a class="anchor" id="id_2a"></a> 

In [10]:
list_of_folders = [units_folder, boron_dlc_folder, avarice_dlc_folder, terran_dlc_folder,
                    split_dlc_folder, timelines_dlc_folder, storage_files, split_storage, avarice_storage,
                    boron_storage,terran_storage, timelines_storage]
list_of_files = [] # list of xml files
list_of_XMLfiles = [] #list of filenames
for folder in list_of_folders:
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.endswith(".xml"):
                list_of_XMLfiles.append( file)
                #print(os.path.join(root, file))
                list_of_files.append(os.path.join(root, file))
print('List of folders in units_folder: ', os.listdir(units_folder))

List of folders in units_folder:  ['size_l', 'size_m', 'size_s', 'size_xl', 'size_xs', 'xref_parts']


### Создание списка путей к конфигурационным файлам кораблей <a class="anchor" id="id_2b_1"></a>

Needed tags: 
```
<explosiondamage shield="5000" value="1000"/>
<storage missile="160" unit="10"/>
<hull max="93000"/>
<secrecy level="2"/>
<purpose primary="fight"/>
<people capacity="44"/>
<shipdetail ref="shipdetail_ship_l_01"/>
<physics mass="196.016">
<inertia pitch="96.271" roll="77.016" yaw="96.271"/>
<drag forward="99.004" horizontal="73.005" pitch="106.203" reverse="396.016" roll="106.203" vertical="73.005" yaw="106.203"/>
</physics>
<thruster tags="large"/>
<ship type="destroyer"/>
</properties> 
``` 
- Equipment 
```
<software>
<software compatible="1" ware="software_dockmk2"/>
<software default="1" ware="software_flightassistmk1"/>
<software default="1" ware="software_scannerlongrangemk1"/>
<software compatible="1" ware="software_scannerlongrangemk2"/>
<software default="1" ware="software_scannerobjectmk1"/>
<software compatible="1" ware="software_scannerobjectmk2"/>
<software default="1" ware="software_targetmk1"/>
<software compatible="1" ware="software_trademk1"/>
</software> 
```
```
      
<connection ref="con_storage01">  # ссылка на файл, содержащий значение емкости грузового отсека
    <macro ref="storage_arg_l_destroyer_01_a_macro" connection="ShipConnection" />
</connection>   
    ```

List of tags containing the information about ship parameters to extract:

In [11]:
needed_tags = ['macro', 'explosiondamage', 'storage', 'hull', 'secrecy', 'purpose', 'people', 'physics', 'inertia', 'drag', 'ship']

In [12]:
list_of_ships = [] # List of files wiht ship data
list_of_ships_configs = [] #List of files containg ship's geometry data needed to calculate equipment slots
list_of_storage_files = [] # files with ship's storage data

keywords = ['ship','macro','spacesuit','cockpit','storage', 'thruster']
for file in list_of_files:
    if keywords[0] in file and keywords[1] in file and keywords[2] not in file and keywords[4] not in file:
        list_of_ships.append(file)
    if  'storage' in file:
        list_of_storage_files.append(file)
    if (
        keywords[0] in file and keywords[1] not in file and keywords[3] not in file and keywords[4] not in file 
                and keywords[5] not in file 
    ):
        list_of_ships_configs.append(file)

print(len(list_of_ships))     
print(len(list_of_ships_configs))  

374
302


### Creating the list of ship names <a class="anchor" id="id_2c"></a>

0001-l044.xml - file with textual information for english version.
The information is divided into pages, each of them contains multiple ids. For example the string  
  
```<page id="20101" title="Ships" descr="Names and descriptions of ships" voice="yes"> ```  
    contains the page id responsible for ship names in the form of {pade_id, ship_id}. The ship Albatros Vanguard have name="{20101,21102}:

```<identification name="{20101,21102}" basename="{20101,21101}" description="{20101,21112}" variation="{20111,1101}" shortvariation="{20111,1103}" icon="ship_xl_build_01" />```  
where  
 20101 - page id, 21102 - id on that page:  
 
```<t id="21101">Albatross</t>```</br>
```<t id="21102">(Albatross Vanguard){20101,21101} {20111,1101}</t>```

In [13]:
with open(descriptions_eng, 'r') as user_file:
    file_contents = user_file.read()
#Create Soup-object:     
descriptions_Sobject = BeautifulSoup(file_contents, features="xml")
list_of_tags_in_textfile = []
for tag in descriptions_Sobject.findAll('page', attrs = {'id':20101,'title':'Ships','descr':'Names and descriptions of ships'}):    
    list_of_tags_in_textfile.append(tag)   


### Creating the table with ship stats <a class="anchor" id="id_2d"></a>

#### Constructing the table <a class="anchor" id="id_2d_1"></a>

Пропуски в количественных данных заменю нулями, так как их наличие означает  отсутствие этого параметра (например ракет у кораблей майнеров).
The missed data in quantative parameters are filled with nulls as they mean that a ship does not have that property.

In [14]:
def file_validation(file):
    """Function to check if file's soup object contains all the needed tags"""
    with open(file, 'r') as user_file:
            file_contents = user_file.read()             
            soup = BeautifulSoup(file_contents, features="xml")            
            if soup.find('properties') and soup.find('identification') and soup.find('physics'):
                 return True
            else:            
                return False

temp = [] # temporary list of lists with ship data needed to construct dataframe from
error_files = []
text_ids_list = [] # List of dictionaries with page_id и name_id needed to merge information about ship names
print(len(list_of_ships))
for file in list_of_ships:
    if not file_validation(file):
            error_files.append(file) 
    else:
             
        ship_data = {}
        with open(file, 'r') as user_file:
            file_contents = user_file.read()             
            soup = BeautifulSoup(file_contents, features="xml")           
            ship_data['component'] = soup.select_one('component').get('ref')
            ship_data['filename'] = soup.select_one('macro').get('name')
            ship_data['ship_class'] = soup.select_one('macro').get('class')
            # name ids
            #ship_data['name_ids'] = soup.select_one('identification').get('name')
           
            t = soup.select_one('identification').get('name')
            ship_data['name_page_id'] = re.findall('\d+',t)[0]
            ship_data['name_id'] = re.findall('\d+',t)[1]
            D = {}                    
            D['pageID'] = ship_data['name_page_id']
            D['nameID'] = ship_data['name_id']
            text_ids_list.append(D)
                
            # race
            ship_data['maker'] = soup.select_one('identification').get('makerrace')
            # storage info
            flag = soup.find('storage')
            if flag:
                ship_data['storage_missile'] = soup.select_one('storage').get('missile')
                ship_data['storage_unit'] = soup.select_one('storage').get('unit')
            else:
                ship_data['storage_missile'] = 0 
                ship_data['storage_unit'] = 0
            # hull info
            flag = soup.find('hull')
            if flag:
                ship_data['hull'] = soup.select_one('hull').get('max')
            else:
                ship_data['hull'] = 0
            # purpose
            flag = soup.find('purpose')
            if flag:
                ship_data['purpose'] = soup.select_one('purpose').get('primary')
            else:
                ship_data['purpose'] = ""
            # people capacity
            flag = soup.find('people')
            if flag:
                ship_data['people_capacity'] = soup.select_one('people').get('capacity')
            else:
                ship_data['people_capacity'] = 0
            # mass
            flag = soup.find('physics')
            if flag:
                ship_data['ship_mass'] = soup.select_one('physics').get('mass')
            else:
                ship_data['ship_mass'] = 0
            # inertia
            flag = soup.find('inertia')
            if flag:
                ship_data['inertia_pitch'] = soup.select_one('inertia').get('pitch')
                ship_data['inertia_yaw'] = soup.select_one('inertia').get('yaw')
                ship_data['inertia_roll'] = soup.select_one('inertia').get('roll')
            else:
                ship_data['inertia_pitch'] = 0
                ship_data['inertia_yaw'] = 0
                ship_data['inertia_roll'] = 0
            # drag
            flag = soup.find('drag')
            if flag:
                ship_data['drag_forward'] = soup.select_one('drag').get('forward')
                ship_data['drag_reverse'] = soup.select_one('drag').get('reverse')
                ship_data['drag_horizontal'] = soup.select_one('drag').get('horizontal')   
                ship_data['drag_vertical'] = soup.select_one('drag').get('vertical')
                ship_data['drag_pitch'] = soup.select_one('drag').get('pitch')
                ship_data['drag_yaw'] = soup.select_one('drag').get('yaw') 
                ship_data['drag_roll'] = soup.select_one('drag').get('roll')
            else:
                ship_data['drag_forward'] = 0
                ship_data['drag_reverse'] = 0
                ship_data['drag_horizontal'] = 0  
                ship_data['drag_vertical'] = 0
                ship_data['drag_pitch'] = 0
                ship_data['drag_yaw'] = 0
                ship_data['drag_roll'] = 0
            # type
            flag = soup.find('ship')
            if flag:
                ship_data['ship_type'] = soup.select_one('ship').get('type')
            else:
                ship_data['ship_type'] = "" 
            # link to storage file            
            flag = soup.find('connection', attrs = {'ref':re.compile("^con_storage")})                        
            if flag:
                 ship_data['storage_file'] = flag.find('macro').get('ref')                    
            else:                    
                 ship_data['storage_file']=""                                
            
            temp.append(ship_data)

374


In [15]:
## Creating table    
ship_data_df = pd.DataFrame.from_dict(temp)
display(ship_data_df.head())

,component,filename,ship_class,name_page_id,name_id,maker,storage_missile,storage_unit,hull,purpose,people_capacity,ship_mass,inertia_pitch,inertia_yaw,inertia_roll,drag_forward,drag_reverse,drag_horizontal,drag_vertical,drag_pitch,drag_yaw,drag_roll,ship_type,storage_file
0,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_a_macro,ship_l,20101,11002,argon,160,10,93000,fight,44,196.016,96.271,96.271,77.016,99.004,396.016,73.005,73.005,106.203,106.203,106.203,destroyer,storage_arg_l_destroyer_01_a_macro
1,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_b_macro,ship_l,20101,11003,argon,160,10,111000,fight,36,235.22,103.378,103.378,82.702,108.805,435.22,87.605,87.605,114.044,114.044,114.044,destroyer,storage_arg_l_destroyer_01_b_macro
2,ship_arg_l_destroyer_02,ship_arg_l_destroyer_02_a_macro,ship_l,20101,11004,argon,160,10,102000,fight,48,260.472,143.455,143.455,114.764,80.583,460.472,43.26,43.26,119.094,119.094,119.094,destroyer,storage_arg_l_destroyer_02_a_macro
3,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_a_macro,ship_l,20101,11104,argon,30,10,26000,mine,46,205.27,133.749,133.749,106.999,56.738,324.216,126.666,126.666,140.897,140.897,140.897,largeminer,storage_arg_l_miner_liquid_01_a_macro
4,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_b_macro,ship_l,20101,11105,argon,30,10,32000,mine,38,246.324,147.778,147.778,118.223,62.485,357.059,151.999,151.999,155.677,155.677,155.677,largeminer,storage_arg_l_miner_liquid_01_b_macro


Add storage value to the table:

In [16]:
storage_list = [] # List with ship's storage data to construct dataframe from
for file in list_of_storage_files:    
    with open(file, 'r') as user_file:
        file_contents = user_file.read()             
        soup = BeautifulSoup(file_contents, features="xml")
        flag =  soup.find('cargo')
        if flag:
            storage_data = {}
            storage_data['file'] = soup.select_one('macro').get('name')
            storage_data['cargo_volume'] = soup.select_one('cargo').get('max')
            storage_data['storage_type'] = soup.select_one('cargo').get('tags')
            storage_list.append(storage_data)
        else:            
            continue


In [17]:
## Creating table    
storage_data_df = pd.DataFrame.from_dict(storage_list)
# dictionary for mapping values into ship_data table
storage_dict = dict(zip(storage_data_df.iloc[:,0],storage_data_df.iloc[:,1]))
display(storage_data_df.sort_values('file').head())
print(storage_data_df['storage_type'].unique())

,file,cargo_volume,storage_type
0,storage_arg_l_destroyer_01_a_macro,2300,container
1,storage_arg_l_destroyer_01_b_macro,2760,container
125,storage_arg_l_destroyer_02_a_macro,3100,container
2,storage_arg_l_miner_liquid_01_a_macro,42000,liquid
3,storage_arg_l_miner_liquid_01_b_macro,50400,liquid


['container' 'liquid' 'solid' 'container liquid solid'
 'container solid liquid' 'container condensate' 'condensate']


In [18]:
ship_data_df = ship_data_df.merge(storage_data_df, how = 'left', 
                                  left_on =['storage_file'], right_on = ['file'],
                                  suffixes=(None,'_y'))


In [19]:
ship_data_df.head()

,component,filename,ship_class,name_page_id,name_id,maker,storage_missile,storage_unit,hull,purpose,people_capacity,ship_mass,inertia_pitch,inertia_yaw,inertia_roll,drag_forward,drag_reverse,drag_horizontal,drag_vertical,drag_pitch,drag_yaw,drag_roll,ship_type,storage_file,file,cargo_volume,storage_type
0,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_a_macro,ship_l,20101,11002,argon,160,10,93000,fight,44,196.016,96.271,96.271,77.016,99.004,396.016,73.005,73.005,106.203,106.203,106.203,destroyer,storage_arg_l_destroyer_01_a_macro,storage_arg_l_destroyer_01_a_macro,2300,container
1,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_b_macro,ship_l,20101,11003,argon,160,10,111000,fight,36,235.22,103.378,103.378,82.702,108.805,435.22,87.605,87.605,114.044,114.044,114.044,destroyer,storage_arg_l_destroyer_01_b_macro,storage_arg_l_destroyer_01_b_macro,2760,container
2,ship_arg_l_destroyer_02,ship_arg_l_destroyer_02_a_macro,ship_l,20101,11004,argon,160,10,102000,fight,48,260.472,143.455,143.455,114.764,80.583,460.472,43.26,43.26,119.094,119.094,119.094,destroyer,storage_arg_l_destroyer_02_a_macro,storage_arg_l_destroyer_02_a_macro,3100,container
3,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_a_macro,ship_l,20101,11104,argon,30,10,26000,mine,46,205.27,133.749,133.749,106.999,56.738,324.216,126.666,126.666,140.897,140.897,140.897,largeminer,storage_arg_l_miner_liquid_01_a_macro,storage_arg_l_miner_liquid_01_a_macro,42000,liquid
4,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_b_macro,ship_l,20101,11105,argon,30,10,32000,mine,38,246.324,147.778,147.778,118.223,62.485,357.059,151.999,151.999,155.677,155.677,155.677,largeminer,storage_arg_l_miner_liquid_01_b_macro,storage_arg_l_miner_liquid_01_b_macro,50400,liquid


Create column with ship names:

In [20]:
ship_names = [] # list with ship names to add it as a column the ship_data_df
# iterate through text_ids_list with page_id and name_id and extract the appropriate ship name and add it to he list:
for element in text_ids_list:
    try:
        ex = descriptions_Sobject.find('page', attrs = {'id':element['pageID'], 'title':'Ships'}).find('t', attrs = {'id':element['nameID']})
        name = re.search('\D+',ex.text).group()
        name = re.split('[^A-Za-z]',name)
        name = list(filter(None, name))
        new_name = " ".join([ele for ele in name if  ele[0].isupper()])       
    except:
        #print(f'Unexpected {err=}, {type(err)=}', element)
        new_name = ""
    finally:
        ship_names.append(new_name)   
  

Check the length of the list and the dataframe then add:

In [21]:
if len(ship_data_df)==len(ship_names):    
    ship_data_df['name'] = ship_names
else:
    print('Lengths  are different')
display(ship_data_df.head())

,component,filename,ship_class,name_page_id,name_id,maker,storage_missile,storage_unit,hull,purpose,people_capacity,ship_mass,inertia_pitch,inertia_yaw,inertia_roll,drag_forward,drag_reverse,drag_horizontal,drag_vertical,drag_pitch,drag_yaw,drag_roll,ship_type,storage_file,file,cargo_volume,storage_type,name
0,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_a_macro,ship_l,20101,11002,argon,160,10,93000,fight,44,196.016,96.271,96.271,77.016,99.004,396.016,73.005,73.005,106.203,106.203,106.203,destroyer,storage_arg_l_destroyer_01_a_macro,storage_arg_l_destroyer_01_a_macro,2300,container,Behemoth Vanguard
1,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_b_macro,ship_l,20101,11003,argon,160,10,111000,fight,36,235.22,103.378,103.378,82.702,108.805,435.22,87.605,87.605,114.044,114.044,114.044,destroyer,storage_arg_l_destroyer_01_b_macro,storage_arg_l_destroyer_01_b_macro,2760,container,Behemoth Sentinel
2,ship_arg_l_destroyer_02,ship_arg_l_destroyer_02_a_macro,ship_l,20101,11004,argon,160,10,102000,fight,48,260.472,143.455,143.455,114.764,80.583,460.472,43.26,43.26,119.094,119.094,119.094,destroyer,storage_arg_l_destroyer_02_a_macro,storage_arg_l_destroyer_02_a_macro,3100,container,Behemoth E
3,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_a_macro,ship_l,20101,11104,argon,30,10,26000,mine,46,205.27,133.749,133.749,106.999,56.738,324.216,126.666,126.666,140.897,140.897,140.897,largeminer,storage_arg_l_miner_liquid_01_a_macro,storage_arg_l_miner_liquid_01_a_macro,42000,liquid,Magnetar Gas Vanguard
4,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_b_macro,ship_l,20101,11105,argon,30,10,32000,mine,38,246.324,147.778,147.778,118.223,62.485,357.059,151.999,151.999,155.677,155.677,155.677,largeminer,storage_arg_l_miner_liquid_01_b_macro,storage_arg_l_miner_liquid_01_b_macro,50400,liquid,Magnetar Gas Sentinel


In [22]:
ship_data_df[ship_data_df['name']=="Asgard"]

,component,filename,ship_class,name_page_id,name_id,maker,storage_missile,storage_unit,hull,purpose,people_capacity,ship_mass,inertia_pitch,inertia_yaw,inertia_roll,drag_forward,drag_reverse,drag_horizontal,drag_vertical,drag_pitch,drag_yaw,drag_roll,ship_type,storage_file,file,cargo_volume,storage_type,name
284,ship_atf_xl_battleship_01,ship_atf_xl_battleship_01_a_macro,ship_xl,20101,66601,terran,1160,10,275000,fight,360,487.388,485.311,485.311,388.249,133.108,532.433,259.232,259.232,717.73,717.73,717.73,battleship,storage_atf_xl_battleship_01_a_macro,storage_atf_xl_battleship_01_a_macro,9000,container,Asgard


#### Check for missed data and clearance <a class="anchor" id="id_2d_2"></a>

#### Change type of columns:

In [23]:
#Изменение типа колонок на float
cols_to_float = ['ship_mass',	'inertia_pitch',	'inertia_yaw',	'inertia_roll',	'drag_forward',
                 'drag_reverse',	'drag_horizontal',	'drag_vertical',	'drag_pitch',	'drag_yaw',	'drag_roll'	]
try:
    for column in cols_to_float:
        ship_data_df[column] = ship_data_df[column].fillna(0)
        ship_data_df[column] = ship_data_df[column].astype('float')

except BaseException:
    print(column)
#Изменение типа колонок на int
cols_to_int = ['storage_missile', 'storage_unit',	'hull',	'people_capacity', 'name_page_id', 'name_id','cargo_volume']
try:
    for column in cols_to_int:
        ship_data_df[column] = ship_data_df[column].fillna(0)        
        ship_data_df[column] = ship_data_df[column].astype('int')
except BaseException:
    print(column)

In [24]:
ship_data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 28 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   component        350 non-null    object 
 1   filename         350 non-null    object 
 2   ship_class       350 non-null    object 
 3   name_page_id     350 non-null    int32  
 4   name_id          350 non-null    int32  
 5   maker            326 non-null    object 
 6   storage_missile  350 non-null    int32  
 7   storage_unit     350 non-null    int32  
 8   hull             350 non-null    int32  
 9   purpose          350 non-null    object 
 10  people_capacity  350 non-null    int32  
 11  ship_mass        350 non-null    float64
 12  inertia_pitch    350 non-null    float64
 13  inertia_yaw      350 non-null    float64
 14  inertia_roll     350 non-null    float64
 15  drag_forward     350 non-null    float64
 16  drag_reverse     350 non-null    float64
 17  drag_horizontal 

Delete unneeded information got to the table:

In [25]:
ship_data_df = ship_data_df.query('name_page_id==20101')

Display the information about missed data:

In [26]:
missing_values_tab(ship_data_df)

Dataframe contains 28 columns and 348 strings.

It has  3 columns with missed values


,missed_values,%_of_all,data_type
file,103,29.60,object
storage_type,103,29.60,object
maker,22,6.32,object


Misses in categorical data are changed to "unknown"

In [27]:
ship_data_df['maker'] = ship_data_df['maker'].fillna('unknown')
display(ship_data_df.head(2))

,component,filename,ship_class,name_page_id,name_id,maker,storage_missile,storage_unit,hull,purpose,people_capacity,ship_mass,inertia_pitch,inertia_yaw,inertia_roll,drag_forward,drag_reverse,drag_horizontal,drag_vertical,drag_pitch,drag_yaw,drag_roll,ship_type,storage_file,file,cargo_volume,storage_type,name
0,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_a_macro,ship_l,20101,11002,argon,160,10,93000,fight,44,196.016,96.271,96.271,77.016,99.004,396.016,73.005,73.005,106.203,106.203,106.203,destroyer,storage_arg_l_destroyer_01_a_macro,storage_arg_l_destroyer_01_a_macro,2300,container,Behemoth Vanguard
1,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_b_macro,ship_l,20101,11003,argon,160,10,111000,fight,36,235.220,103.378,103.378,82.702,108.805,435.220,87.605,87.605,114.044,114.044,114.044,destroyer,storage_arg_l_destroyer_01_b_macro,storage_arg_l_destroyer_01_b_macro,2760,container,Behemoth Sentinel


## Get the data about ship equipment <a class="anchor" id="id_3"></a>

### How find data about the number of equipment slots  <a class="anchor" id="id_2f"></a>

The information about ship shields contained in files with ship's geometry:  
h:/Steam/X4/Unpacked/assets/units/size_xl/ship_arg_xl_carrier_01.xml  
```  
 <connection name="con_shieldgen_xl_01" tags="extralarge shield standard ">
				<offset>
					<position x="179.2407" y="140.9466" z="-461.2684"/>
				</offset>
			</connection>  
```

In [28]:
with open(r'h:/Steam/X4/Unpacked/assets/units/size_xl/ship_arg_xl_carrier_01.xml', 'r') as user_file:
    file_contents = user_file.read()
#Создание древовидной структуры файла      
Sobject = BeautifulSoup(file_contents, features="xml")
tags = Sobject.findAll('connection', attrs = {'tags':"extralarge shield standard "})
print("The number of XL shields: ",len(tags))

The number of XL shields:  3


Calculate the number of countermeasures:

In [29]:
tags = Sobject.findAll('connection', attrs = {'tags':"countermeasures "})
print("Number of countermeasures: ",len(tags))

Number of countermeasures:  16


Calculate the number of turrets on M ships:

In [30]:
tags="turret medium standard platformcollision unhittable  combat " # attribute value use to find the number of its slots
tags_e = "engine medium  standard " # engine attribute
tags_s="medium shield unhittable platformcollision standard " # shields on turrets attribute
with open(r'h:\Steam\X4\Unpacked\assets\units\size_m\ship_arg_m_frigate_01.xml', 'r') as user_file:
    file_contents = user_file.read()
#Creating Soup object of the file      
Sobject = BeautifulSoup(file_contents, features="xml")
t = Sobject.findAll('connection', attrs = {'tags':tags})
e = Sobject.findAll('connection', attrs = {'tags':tags_e})
s = Sobject.findAll('connection', attrs = {'tags':tags_s})
print("The number of turret slots: ",len(t))
print("The number of engine slots: ",len(e))
print("The number of shield slots: ",len(s))

The number of turret slots:  4
The number of engine slots:  2
The number of shield slots:  3


Find the number of engines on S ships:

In [31]:
tag = "engine small platformcollision  standard "
with open(r'h:\Steam\X4\Unpacked\assets\units\size_s\ship_arg_s_fighter_01.xml', 'r') as user_file:
    file_contents = user_file.read()
#Создание древовидной структуры файла      
Sobject = BeautifulSoup(file_contents, features="xml")
t = Sobject.findAll('connection', attrs = {'tags':tag})
print(len(t))

2


The algorithm of data gathering:
1. Iterate through folders in ```units_folder``` except for folders xref_parts as they contain information about civilian objects (drones, police, escapepods etc).
2. Find xml files with "ship" in their name.
3. Create beatifulsoup object of each file.    
4. Do counting

With the help of component attribute we can link xml files in situated in '../units/size_s' folder  
```<component name="ship_arg_l_destroyer_01" class="ship_l">```   
with xml files with ship data in '../units/size_s/macros/' folder:    
```<macros>
  <macro name="ship_arg_l_destroyer_01_a_macro" class="ship_l">
    <component ref="ship_arg_l_destroyer_01" />
    ```            
And add data about slots in ship_data table.

Equipment tag names in XML-files:

In [32]:
# Boron ships
boron_shield_tags = ["boron extralarge shield", # XL shields
                     "boron large shield",  # L shields
                     "boron hittable medium shield", # M shields on turrets of L/XL shields
                     "boron medium shield unhittable", #  M shield on M ships
                     "boron shield small unhittable" # S shields
                     ]
boron_turret_tags = ["boron combat large missile turret",   #  L turrets                
                     "boron combat hittable medium missile turret", # M turrets
                     "boron combat medium turret unhittable",  # M turrets on M ships
                     "boron large mining turret" # L turrets on boron L solid miner ships
                     ]
boron_engine_tags = ["boron engine extralarge", "boron engine large", "boron engine medium", "boron engine small"]

# Ships of other factions
shield_tags = ["extralarge shield standard", # XL shields
                "large shield  standard",  # L shields
                "medium shield hittable  standard", #M shields on turrets of L/XL shields
                "medium shield unhittable platformcollision standard", # M shield on M ships 
                "small shield unhittable  standard" # S shields  
                ]
turret_tags = ["turret large standard missile  combat", #L turrets on  XL/L ships
               "turret medium standard missile hittable  combat", # M turrets on  XL/L ships
                "turret medium standard platformcollision unhittable  combat" # М turrets on M ships
                "turret large mining standard" #  L turrets on L solid miner ships 
               ]
engine_tags = ["engine extralarge standard", # XL engines
               "engine large  standard",  # L engines
                 "engine medium  standard",  # M engines
                 "engine small platformcollision  standard" # S engines
                 ]
weapon_tags = ["weapon large arg_destroyer_01 ", # primary weapons tag
               "weapon medium standard missile platformcollision symmetry symmetry_right combat ", # M weapons tag
               "weapon small standard missile platformcollision symmetry symmetry_right  combat ", # S weapons tag
               "weapon large ter_destroyer_01 symmetry symmetry_1 symmetry_right", # primary weapons tag on terran ships
               "weapon large ter_destroyer_01 symmetry symmetry_1 symmetry_left", 
               "weapon mandatory extralarge atf_battleship_01 ", # primary weapon on Asgarth
               "weapon extralarge pir_battleship_01 "]    # primary weapon tag on Earlking

In [33]:
ship_config = {}  # The dictionary of dictionaries where key is the component name and its value is the dictionary with extracted information
for file in list_of_ships_configs:
    with open(file, 'r') as user_file:
        file_contents = user_file.read()
         #create a beatiful soup object      
        file_Sobject = BeautifulSoup(file_contents, features="xml")
        

        file_data = {} # dictionary for store information about equipment slots
        # Get the component name
        component =  file_Sobject.select_one('component').get('name').strip()
        file_data['class'] = file_Sobject.select_one('component').get('class')
        # Count the number of equipment slots
        try:
           # if ship is XL ship and not an L terran liquid miner(because of dev's error)
            if  file_data['class'] =='ship_xl' and component!='ship_ter_l_miner_liquid_01':
               #XL shields
               tags_xls = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*extralarge)(?=.*shield)')})
               file_data['primary_shield_slots'] = len(tags_xls)
               #M shields
               tags_xms = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*medium)(?=.*shield)(?=.*hittable)')})
               file_data['turret_shield_slots'] = len(tags_xms)
               # Count L turrets               
               tags_xlt = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*large)(?=.*turret)')})
               file_data['l_turret_slots'] = len(tags_xlt)
               #Count M turrets
               tags_xmt = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*medium)(?=.*turret)(?=.*hittable)')})
               file_data['m_turret_slots'] = len(tags_xmt)
                #calculate number of primary XL weapon slots
               tags_xlw = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*extralarge)(?=.*weapon)')})
               file_data['primary_weapon_slots'] = len(tags_xlw)
               #Count engines
               tags_xe = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*extralarge)(?=.*engine)')})
               file_data['engines_slots'] = len(tags_xe)
               # Save the info into the main dictionary under the key with ship's component
               ship_config[component] = file_data
            
            elif  file_data['class'] =='ship_xl' and component =='ship_ter_l_miner_liquid_01': # case of Hokkaido Gas miner
                #Count L shields
               tags_ls = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*large)(?=.*shield)')})
               file_data['primary_shield_slots'] = len(tags_ls)
                #count M shields
               tags_lms = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*medium)(?=.*shield)(?=.*hittable)')})
               file_data['turret_shield_slots'] = len(tags_lms)
               # Count L turrets              
               tags_lt = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*large)(?=.*turret)')})
               file_data['l_turret_slots'] = len(tags_lt)
               #count M turrets
               tags_lmt = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*medium)(?=.*turret)(?=.*hittable)')})
               file_data['m_turret_slots'] = len(tags_lmt)
                  #calculate number of primary L weapon slots
               tags_lw = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*large)(?=.*weapon)')})
               file_data['primary_weapon_slots'] = len(tags_lw)
               #count engines
               tags_le = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*large)(?=.*engine)')})
               file_data['engines_slots'] = len(tags_le)
               ship_config[component] = file_data
             
             # For L ships            
            if  file_data['class'] =='ship_l':               
               #Count L shields
               tags_ls = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*large)(?=.*shield)')})
               file_data['primary_shield_slots'] = len(tags_ls)
                #Count M shields
               tags_lms = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*medium)(?=.*shield)(?=.*hittable)')})
               file_data['turret_shield_slots'] = len(tags_lms)
               # Count L turrets             
               tags_lt = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*large)(?=.*turret)')})
               file_data['l_turret_slots'] = len(tags_lt)
               #count M turrets
               tags_lmt = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*medium)(?=.*turret)(?=.*hittable)')})
               file_data['m_turret_slots'] = len(tags_lmt)
                  #calculate number of primary L weapon slots
               tags_lw = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*large)(?=.*weapon)')})
               file_data['primary_weapon_slots'] = len(tags_lw)
               #Count engines
               tags_le = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*large)(?=.*engine)')})
               file_data['engines_slots'] = len(tags_le)
               ship_config[component] = file_data
               
               # For M ships
            if  file_data['class'] =='ship_m':               
                #Count M shields
               tags_ms = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*medium)(?=.*shield)(?=.*unhittable)')})
               file_data['primary_shield_slots'] = len(tags_ms) 
               # The number of M shields on turrets is 0
               file_data['turret_shield_slots'] = 0
               # the number of L turrets is 0
               file_data['l_turret_slots'] = 0          
               #Count M turrets
               tags_mt = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*medium)(?=.*turret)(?=.*unhittable)')})
               file_data['m_turret_slots'] = len(tags_mt)
                #calculate number of primary M weapon slots
               tags_mw = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*medium)(?=.*weapon)')})
               file_data['primary_weapon_slots'] = len(tags_mw)
               #Count engines
               tags_me = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*medium)(?=.*engine)')})
               file_data['engines_slots'] = len(tags_me)
               ship_config[component] = file_data
             
             # For s ships
            if  file_data['class'] =='ship_s':
               # count shields
               tags_ss = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*small)(?=.*shield).*$')})
               file_data['primary_shield_slots'] = len(tags_ss) 
                  #calculate number of primary S weapon slots
               tags_sw = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*small)(?=.*weapon)')})
               file_data['primary_weapon_slots'] = len(tags_sw)              
               #count engines
               tags_se = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*small)(?=.*engine).*$')})
               file_data['engines_slots'] = len(tags_se) 
               # set other parameters to 0
               file_data['turret_shield_slots'] = 0
               file_data['l_turret_slots'] = 0
               file_data['m_turret_slots'] = 0
               # save the gathered data into main dictionary
               ship_config[component] = file_data
            # if it is an NPC ship
            if  file_data['class'] =='ship_xs':                            
               #they have only engines
               tags_se = file_Sobject.findAll('connection', attrs = {'tags':re.compile('^(?=.*small)(?=.*engine).*$')})
               file_data['engines_slots'] = len(tags_se) 
               file_data['primary_shield_slots'] = 0 
               file_data['turret_shield_slots'] = 0
               file_data['l_turret_slots'] = 0
               file_data['m_turret_slots'] = 0
               file_data['primary_weapon_slots'] = 0
               ship_config[component] = file_data

        except:           
           print('error!!!',file)
#print(ship_config)

Creating the dataframe from the dictionary:

In [34]:
ship_configs_df = pd.DataFrame.from_dict(ship_config,orient = 'index').reset_index().rename(columns = {'index':'component'})
display(ship_configs_df.head())


,component,class,primary_shield_slots,turret_shield_slots,l_turret_slots,m_turret_slots,primary_weapon_slots,engines_slots
0,ship_arg_l_destroyer_01,ship_l,3,9,2,8,2,3
1,ship_arg_l_destroyer_02,ship_l,3,9,2,8,2,3
2,ship_arg_l_miner_liquid_01,ship_l,2,7,0,6,0,2
3,ship_arg_l_miner_solid_01,ship_l,2,9,1,6,0,2
4,ship_arg_l_trans_container_01,ship_l,2,5,0,7,0,2


Check for missing data:

In [35]:
ship_configs_df[ship_configs_df.isna().any(axis = 1)]

,component,class,primary_shield_slots,turret_shield_slots,l_turret_slots,m_turret_slots,primary_weapon_slots,engines_slots


##### Merge ship_data and ship_configs

In [36]:
ship_data_df = ship_data_df.merge(ship_configs_df, on = 'component', how = 'left', suffixes=(None,'_y'))
# Deleting duplicated column
ship_data_df = ship_data_df.drop('class', axis =1)
#Fill missed values with 0 as it means no slots
ship_data_df = ship_data_df.fillna(0)

In [37]:
#Changing data types of columns
cols_to_int = ['primary_shield_slots',	'turret_shield_slots',	'l_turret_slots',	'm_turret_slots',
               	'engines_slots', 'primary_weapon_slots']
try:
    for column in cols_to_int:
        ship_data_df[column] = ship_data_df[column].fillna(0)        
        ship_data_df[column] = ship_data_df[column].astype('int')
except BaseException:
    print(column)
ship_data_df.head()

,component,filename,ship_class,name_page_id,name_id,maker,storage_missile,storage_unit,hull,purpose,people_capacity,ship_mass,inertia_pitch,inertia_yaw,inertia_roll,drag_forward,drag_reverse,drag_horizontal,drag_vertical,drag_pitch,drag_yaw,drag_roll,ship_type,storage_file,file,cargo_volume,storage_type,name,primary_shield_slots,turret_shield_slots,l_turret_slots,m_turret_slots,primary_weapon_slots,engines_slots
0,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_a_macro,ship_l,20101,11002,argon,160,10,93000,fight,44,196.016,96.271,96.271,77.016,99.004,396.016,73.005,73.005,106.203,106.203,106.203,destroyer,storage_arg_l_destroyer_01_a_macro,storage_arg_l_destroyer_01_a_macro,2300,container,Behemoth Vanguard,3,9,2,8,2,3
1,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_b_macro,ship_l,20101,11003,argon,160,10,111000,fight,36,235.220,103.378,103.378,82.702,108.805,435.220,87.605,87.605,114.044,114.044,114.044,destroyer,storage_arg_l_destroyer_01_b_macro,storage_arg_l_destroyer_01_b_macro,2760,container,Behemoth Sentinel,3,9,2,8,2,3
2,ship_arg_l_destroyer_02,ship_arg_l_destroyer_02_a_macro,ship_l,20101,11004,argon,160,10,102000,fight,48,260.472,143.455,143.455,114.764,80.583,460.472,43.260,43.260,119.094,119.094,119.094,destroyer,storage_arg_l_destroyer_02_a_macro,storage_arg_l_destroyer_02_a_macro,3100,container,Behemoth E,3,9,2,8,2,3
3,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_a_macro,ship_l,20101,11104,argon,30,10,26000,mine,46,205.270,133.749,133.749,106.999,56.738,324.216,126.666,126.666,140.897,140.897,140.897,largeminer,storage_arg_l_miner_liquid_01_a_macro,storage_arg_l_miner_liquid_01_a_macro,42000,liquid,Magnetar Gas Vanguard,2,7,0,6,0,2
4,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_b_macro,ship_l,20101,11105,argon,30,10,32000,mine,38,246.324,147.778,147.778,118.223,62.485,357.059,151.999,151.999,155.677,155.677,155.677,largeminer,storage_arg_l_miner_liquid_01_b_macro,storage_arg_l_miner_liquid_01_b_macro,50400,liquid,Magnetar Gas Sentinel,2,7,0,6,0,2


In [38]:
missing_values_tab(ship_data_df)

Dataframe contains 34 columns and 348 strings.

It has  0 columns with missed values


,missed_values,%_of_all,data_type


#### Shields <a class="anchor" id="id_3a"></a>

Ship "Behemot Vanguard" , Id = 11002

Page id of shields in descriptions_eng  
```<page id="20106" title="Shields" descr="Names and descriptions of ship and station shields" voice="no">```

Example of an shield xml file (shield_arg_l_standard_01_mk1_macro.xml):  
```
<macros>
  <macro name="shield_arg_l_standard_01_mk1_macro" class="shieldgenerator">
    <component ref="shield_arg_l_standard_01_mk1" />
    <properties>
      <identification name="{20106,3004}" basename="{20106,3001}" shortname="{20106,3005}" makerrace="argon" description="{20106,3002}" mk="1" />
      <recharge max="38844" rate="173" delay="0" />
      <hull max="2000" threshold="0.2" />
    </properties>
  </macro>
</macros>```

The algorithm of getting the data:  
1. Go to the catalog ```cat_shields```
2. Find files with "shield" in their names and store them in a list.
3. Iterate through the list and create  dictionary ```shield_info```.   
4. Make a dataframe from the dictionary.   
As stated on egosoft wiki M shield on turret and engine slots of L/XL ships have the "hull" parameter not equal to null.

##### Create list of shield files

In [39]:
list_of_shield_files = [] # list of files with shield data
list_of_shipstorage_files = [] # list of files with dock capacity data
# List of catalogs to traverse
shield_folders = [cat_shields,split_shields,avarice_shields,boron_shields,terran_shields,timelines_shields ]
for folder in shield_folders:
        for root, dirs, files in os.walk(folder):                
                for file in files:
                        if  file.startswith("shield"):                 
                                #print(os.path.join(root, file))
                                list_of_shield_files.append(os.path.join(root, file))
                        if  file.startswith("shipstorage"):                 
                                #print(os.path.join(root, file))
                                list_of_shipstorage_files.append(os.path.join(root, file))
                        
# Deleting unused files
list_of_shield_files.remove(r'h:\Steam\X4\Unpacked\extensions\avarice_dlc\assets\props\surfaceelements\macros\shield_gen_m_yacht_01_mk1_video_macro.xml')
list_of_shield_files.remove(r'h:\Steam\X4\Unpacked\extensions\boron_dlc\assets\props\surfaceelements\macros\shield_bor_s_standard_01_mk1_video_macro.xml')
list_of_shield_files.remove(r'h:\Steam\X4\Unpacked\extensions\boron_dlc\assets\props\surfaceelements\macros\shield_bor_s_standard_01_mk2_video_macro.xml')
list_of_shield_files.remove(r'h:\Steam\X4\Unpacked\extensions\boron_dlc\assets\props\surfaceelements\macros\shield_bor_s_standard_01_mk3_video_macro.xml')
print(len(list_of_shield_files), len(list_of_shipstorage_files))


110 16


In [40]:
# Beatiful soup object of the part of descriptions eng where page id = 20106 to get shield names
shield_page = descriptions_Sobject.find('page', attrs = {'id':20106})
shield_info = {} # main dict to store data
count = 0 # a variable to count files
#Iterate through list of shield files:
for file in list_of_shield_files:
    with open(file, 'r') as user_file:
        file_contents = user_file.read()            
        shield_Sobject = BeautifulSoup(file_contents, features="xml")
        
        try:
            shield_data = {} # create an empty dictionary to store data from the open file
        # get filename of the file 
            shield_data['filename'] =  shield_Sobject.select_one('macro').get('name')+".xml"
        #  get page id and id
        
            temp =  shield_Sobject.select_one('identification').get('name')
            name = re.search('[^{}]+',temp).group()
            name_list = name.split(',')
            shield_page_id = int(name_list[0]) 
            shield_id = int(name_list[1])       
            shield_data['page_id'] = shield_page_id  
            shield_data['shield_id'] = shield_id     
        except:             
            print('identification name error', file)            
            continue
        # get info about shield maker
        shield_data['maker'] =  shield_Sobject.select_one('identification').get('makerrace')
        # get shield grade
        try:
            shield_data['version'] =  shield_Sobject.select_one('identification').get('mk')
            shield_data['version'] = int(shield_data['version'])
        except:
            print('version error', file)            
        # get info about shield points
        try:
            shield_data['shield_value'] =  shield_Sobject.select_one('recharge').get('max')
            shield_data['shield_value'] = int(shield_data['shield_value'])
        except:
            print('shield value error', file)            
        #get shield regeneration value
        try:
            shield_data['recharge_rate'] =  shield_Sobject.select_one('recharge').get('rate')
            shield_data['recharge_rate'] = int(shield_data['recharge_rate'])
        except:
            print('recharge rate error', file)            
        #get shield delay
        try:
            shield_data['recharge_delay'] =  shield_Sobject.select_one('recharge').get('delay')
            shield_data['recharge_delay'] = float(shield_data['recharge_delay'])
        except:
            print('recharge delay error', file)            
        # get info about shield hitpoints
        try:

            shield_data['hull'] =  shield_Sobject.select_one('hull').get('max')
            shield_data['hull'] = int(shield_data['hull'])
        except:
            shield_data['hull'] = 0
        
        count +=1
#Store shield_data into shield_info under the key of its name got from descriptions_eng
        try:
            tmp = shield_page.find('t',attrs = {'id':shield_data['shield_id']})
            name = re.search('[^()]+',tmp.text).group()   
            shield_info[name] = shield_data
        except:    
            print("Error", shield_data)           

print('Length of shield_info: ', len(shield_info))
print('total files: ',count)

identification name error h:\Steam\X4\Unpacked\assets\props\SurfaceElements\macros\shield_xen_m_standard_02_mk1_macro.xml
identification name error h:\Steam\X4\Unpacked\assets\props\SurfaceElements\macros\shield_xen_m_virtual_01_mk1_video_macro.xml
identification name error h:\Steam\X4\Unpacked\assets\props\SurfaceElements\macros\shield_xen_s_virtual_01_mk1_video_macro.xml
identification name error h:\Steam\X4\Unpacked\extensions\timelines_dlc\assets\props\surfaceelements\macros\shield_ter_m_virtual_01_mk1_video_macro.xml
identification name error h:\Steam\X4\Unpacked\extensions\timelines_dlc\assets\props\surfaceelements\macros\shield_ter_s_virtual_01_mk1_video_macro.xml
Length of shield_info:  76
total files:  105


##### Creating and saving of the shield table

In [41]:
shields_df = pd.DataFrame.from_dict(shield_info,orient = 'index').reset_index().rename(columns = {'index':'name'})
display(shields_df.head())

,name,filename,page_id,shield_id,maker,version,shield_value,recharge_rate,recharge_delay,hull
0,ARG L Shield Generator Mk1,shield_arg_l_standard_01_mk1_macro.xml,20106,3004,argon,1,38844,173,0.0,2000
1,ARG L Shield Generator Mk2,shield_arg_l_standard_01_mk2_macro.xml,20106,3044,argon,2,46282,268,0.0,2000
2,ARG M Shield Generator Mk1,shield_arg_m_standard_02_mk1_macro.xml,20106,2004,argon,1,5147,26,0.5,500
3,ARG M Shield Generator Mk2,shield_arg_m_standard_02_mk2_macro.xml,20106,2044,argon,2,6133,41,0.5,500
4,ARG S Shield Generator Mk1,shield_arg_s_racer_01_mk1_macro.xml,20106,1004,argon,1,827,100,8.0,0


In [42]:
#Show info
missing_values_tab(shields_df)
shields_df.info()


Dataframe contains 10 columns and 76 strings.

It has  1 columns with missed values


,missed_values,%_of_all,data_type
maker,1,1.32,object


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            76 non-null     object 
 1   filename        76 non-null     object 
 2   page_id         76 non-null     int64  
 3   shield_id       76 non-null     int64  
 4   maker           75 non-null     object 
 5   version         76 non-null     int64  
 6   shield_value    76 non-null     int64  
 7   recharge_rate   76 non-null     int64  
 8   recharge_delay  76 non-null     float64
 9   hull            76 non-null     int64  
dtypes: float64(1), int64(6), object(3)
memory usage: 6.1+ KB


In [43]:
display(shields_df.query('maker.isna()'))

,name,filename,page_id,shield_id,maker,version,shield_value,recharge_rate,recharge_delay,hull
67,S Racing Shield Generator Mk1,shield_gen_s_racer_01_mk1_macro.xml,20106,1274,None,1,827,82,12.1,0


The missed row is related to a new ship from the DLC "Timelines", but because it is a racing ship and they are excluded from the analysis it is not important

In [44]:
# проверка на ошибки
display(shields_df.query('maker=="split"'))

,name,filename,page_id,shield_id,maker,version,shield_value,recharge_rate,recharge_delay,hull
35,SPL L Shield Generator Mk1,shield_spl_l_standard_01_mk1_macro.xml,20106,3084,split,1,33018,140,0.00,3000
36,SPL L Shield Generator Mk2,shield_spl_l_standard_01_mk2_macro.xml,20106,3094,split,2,39340,217,0.00,3000
37,SPL M Shield Generator Mk1,shield_spl_m_standard_02_mk1_macro.xml,20106,2084,split,1,4375,21,0.36,500
38,SPL M Shield Generator Mk2,shield_spl_m_standard_02_mk2_macro.xml,20106,2094,split,2,5213,33,0.36,500
39,SPL S Shield Generator Mk1,shield_spl_s_standard_01_mk1_macro.xml,20106,1124,split,1,703,67,8.90,0
40,SPL S Shield Generator Mk2,shield_spl_s_standard_01_mk2_macro.xml,20106,1134,split,2,840,103,8.90,0
41,SPL S Shield Generator Mk3,shield_spl_s_standard_01_mk3_macro.xml,20106,1144,split,3,1200,177,8.90,0
42,SPL XL Shield Generator Mk1,shield_spl_xl_standard_01_mk1_macro.xml,20106,4044,split,1,110058,398,0.00,9000


###  Engines <a class="anchor" id="id_3b"></a>

In [45]:
list_of_engine_files = [] #List of needed files with engines
# List of keywords to ignore files with mines, drones, civilian transport, missiles
keywords_to_ignore = ['_xs_','_spacesuit_','_mine_','_missile_']
# list of engine catalogs
engine_folders = [cat_engines,split_engines,avarice_engines,boron_engines,terran_engines, timelines_engines ]
for folder in engine_folders:
        for root, dirs, files in os.walk(folder):                
                for file in files:
                        if  (file.startswith("engine_") 
                             and '_video_' not in file 
                             and '_xs_' not in file 
                             and '_spacesuit_' not in file 
                             and '_mine_' not in  file
                             and '_missile_' not in file):                 
                                #print(os.path.join(root, file))
                                list_of_engine_files.append(os.path.join(root, file))
print(len(list_of_engine_files))

151


Gathering data about all engines

!!! In place of boron engine in the file 'engine_bor_l_allround_01_mk1_macro.xml' the game use 'BOR L All-round Engine Mk1' from the file 'engine_bor_l_travel_01_mk1_macro.xml'.

In [46]:
#get beatiful soup object with engine names from descriptions_eng with page id = 20107 
engine_page = descriptions_Sobject.find('page', attrs = {'id':20107})
engine_info = {} # main dictionary to store data
#Iterate through needed files
for file in list_of_engine_files:    
    with open(file, 'r') as user_file:
        file_contents = user_file.read()             
        engine_Sobject = BeautifulSoup(file_contents, features="xml") 
        engine_data = {} # create dictionary to store information from the opened file
        try:
            engine_data['filename'] = engine_Sobject.select_one('macro').get('name')+".xml"
        except:
            print('Error with filename', file)
            # get class of engine
        try:
            engine_data['class'] = engine_Sobject.select_one('macro').get('class')
        except:
            print("Error with class", file)            
            engine_data['class']  = 'None'
        #  get page_id и id  descriptions_eng
        try:
            temp =  engine_Sobject.select_one('identification').get('name')
            name = re.search('[^{}]+',temp).group()
            name_list = name.split(',')
            engine_page_id = int(name_list[0]) 
            engine_id = int(name_list[1])      
            engine_data['page_id'] = engine_page_id  
            engine_data['engine_id'] = engine_id               
            if temp == "{0,0,#'engine_bor_l_allround_01_mk1'}":
                    engine_data['page_id'] = 0  
                    engine_data['engine_id'] = 0                    
        except:
            print("Error with page id", file)               
            # maker
        try:
            engine_data['maker'] = engine_Sobject.select_one('identification').get('makerrace')
        except:
            print("Error with maker", file)                
            engine_data['maker'] = 'None'
            # version
        try:
            engine_data['version'] = engine_Sobject.select_one('identification').get('mk')
        except:
            print("Error with version", file)                
            engine_data['version']  = 'None' 
            #  boost   parameters
        try: 
            engine_data['boost_duration'] = engine_Sobject.select_one('boost').get('duration')
            engine_data['boost_thrust'] = engine_Sobject.select_one('boost').get('thrust')
            engine_data['boost_attack'] = engine_Sobject.select_one('boost').get('attack')
            engine_data['boost_release'] = engine_Sobject.select_one('boost').get('release')
        except:
            print('No boost parameter or error', file)                
            engine_data['boost_duration']  = 0
            engine_data['boost_thrust'] = 0
            engine_data['boost_attack'] = 0
            engine_data['boost_release'] = 0
            # travel parameters
        try:
            engine_data['travel_charge'] = engine_Sobject.select_one('travel').get('charge')
            engine_data['travel_thrust'] = engine_Sobject.select_one('travel').get('thrust')
            engine_data['travel_attack'] = engine_Sobject.select_one('travel').get('attack')
            engine_data['travel_release'] = engine_Sobject.select_one('travel').get('release')
        except:
            print("Errors with travel parameters", file)
            engine_data['travel_charge'] = 0
            engine_data['travel_thrust'] = 0
            engine_data['travel_attack'] = 0
            engine_data['travel_release'] = 0
            # thrust parameters
        try:
            engine_data['thrust_forward'] = engine_Sobject.select_one('thrust').get('forward')
            engine_data['thrust_reverse'] = engine_Sobject.select_one('thrust').get('reverse')
        except:
            print("Errors with thrust parameters", file) 
            engine_data['thrust_forward'] = 0
            engine_data['thrust_reverse'] = 0 

            # store "engine_data" dictionary into "engine_info" dictionary under the key of its name got from descriptions_eng
        try:
            if engine_data['page_id']==0 and engine_data['engine_id']==0:  # в случае {0,0,#'engine_bor_l_allround_01_mk1'}
                continue
            elif engine_data['engine_id']==3124:
                name = 'BOR L All-round Engine Mk1'
                engine_info[name] = engine_data 

            else:        
                tmp = engine_page.find('t',attrs = {'id':engine_data['engine_id']})
                name = re.search('[^()]+',tmp.text).group()   
                engine_info[name] = engine_data           
        except:
            print("Errors while assigning to key in engine_info", file)
            
print('Total engines: ', len(engine_info))

No boost parameter or error h:/Steam/X4/Unpacked/assets/props/Engines/macros\engine_kha_l_destroyer_01_allround_01_mk1_macro.xml
Errors with travel parameters h:/Steam/X4/Unpacked/assets/props/Engines/macros\engine_kha_l_destroyer_01_allround_01_mk1_macro.xml
No boost parameter or error h:/Steam/X4/Unpacked/assets/props/Engines/macros\engine_kha_xl_battleship_01_allround_01_mk1_macro.xml
Errors with travel parameters h:/Steam/X4/Unpacked/assets/props/Engines/macros\engine_kha_xl_battleship_01_allround_01_mk1_macro.xml
No boost parameter or error h:\Steam\X4\Unpacked\extensions\timelines_dlc\assets\props\engines\macros\engine_tel_xl_travel_02_mk1_macro.xml
No boost parameter or error h:\Steam\X4\Unpacked\extensions\timelines_dlc\assets\props\engines\macros\engine_xen_xl_mothership_01_allround_mk1_macro.xml
Errors with travel parameters h:\Steam\X4\Unpacked\extensions\timelines_dlc\assets\props\engines\macros\engine_xen_xl_mothership_01_allround_mk1_macro.xml
Total engines:  141


In [47]:
engines_df = pd.DataFrame.from_dict(engine_info,orient = 'index').reset_index().rename(columns = {'index':'name'})
display(engines_df.head())

,name,filename,class,page_id,engine_id,maker,version,boost_duration,boost_thrust,boost_attack,boost_release,travel_charge,travel_thrust,travel_attack,travel_release,thrust_forward,thrust_reverse
0,ARG L All-round Engine Mk1,engine_arg_l_allround_02_mk1_macro.xml,engine,20107,3004,argon,1,59,2,10,1,20,13,750,22.5,4206,4627
1,ARG L Travel Engine Mk1,engine_arg_l_travel_01_mk1_macro.xml,engine,20107,3044,argon,1,56,2,10,1,20,33,85,37.5,4006,3605
2,ARG M All-round Engine Mk1,engine_arg_m_allround_01_mk1_macro.xml,engine,20107,2004,argon,1,7,8,0.25,1,1,9,30,20,1002,952
3,ARG M All-round Engine Mk2,engine_arg_m_allround_01_mk2_macro.xml,engine,20107,2044,argon,2,7,8,0.25,1,1,9,30,20,1212,1228
4,ARG M All-round Engine Mk3,engine_arg_m_allround_01_mk3_macro.xml,engine,20107,2084,argon,3,7,8,0.25,1,1,9,30,20,1353,1413


Checking for duplicate values:

In [48]:
display(engines_df.groupby('name')['name'].value_counts())

name
ARG L All-round Engine Mk1     1
ARG L Travel Engine Mk1        1
ARG M All-round Engine Mk1     1
ARG M All-round Engine Mk2     1
ARG M All-round Engine Mk3     1
                              ..
XEN M Combat Engine Mk1        1
XEN M Travel Engine Mk1        1
XEN S Combat Engine Mk1        1
XEN XL All-round Engine Mk1    1
Xenon XL Engine                1
Name: count, Length: 141, dtype: int64

Checking for missed data:

In [49]:
missing_values_tab(engines_df)
engines_df.info()

Dataframe contains 17 columns and 141 strings.

It has  3 columns with missed values


,missed_values,%_of_all,data_type
maker,3,2.13,object
version,1,0.71,object
thrust_reverse,1,0.71,object


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141 entries, 0 to 140
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   name            141 non-null    object
 1   filename        141 non-null    object
 2   class           141 non-null    object
 3   page_id         141 non-null    int64 
 4   engine_id       141 non-null    int64 
 5   maker           138 non-null    object
 6   version         140 non-null    object
 7   boost_duration  141 non-null    object
 8   boost_thrust    141 non-null    object
 9   boost_attack    141 non-null    object
 10  boost_release   141 non-null    object
 11  travel_charge   141 non-null    object
 12  travel_thrust   141 non-null    object
 13  travel_attack   141 non-null    object
 14  travel_release  141 non-null    object
 15  thrust_forward  141 non-null    object
 16  thrust_reverse  140 non-null    object
dtypes: int64(2), object(15)
memory usage: 18.9+ KB


In [50]:
engines_df[engines_df.isna().any(axis =1)]

,name,filename,class,page_id,engine_id,maker,version,boost_duration,boost_thrust,boost_attack,boost_release,travel_charge,travel_thrust,travel_attack,travel_release,thrust_forward,thrust_reverse
99,SPL XL Travel Engine Mk1,engine_tel_xl_travel_02_mk1_macro.xml,engine,20107,4094,None,None,0,0,0,0,0.1,100,0.1,60.1,900,None
133,S Racing Engine Mk1,engine_gen_s_racer_01_mk1_macro.xml,engine,20107,1914,None,1,7,8,0.25,1,1,14,30,20,396,416
134,S Racing Engine Mk2,engine_gen_s_racer_01_mk2_macro.xml,engine,20107,1924,None,2,6,6.4,0.25,0.5,0,1,1,1,719.5,574


Changing data types

In [51]:
#Lists of columns to change to int or float
cols_to_int = ['boost_duration','travel_charge','travel_thrust','thrust_forward', 'thrust_reverse']
cols_to_float = ['boost_thrust','boost_attack','boost_release', 'travel_attack', 'travel_release']
try:
    for column in cols_to_int:               
        engines_df[column] = engines_df[column].astype('int')
    for column in cols_to_float:               
        engines_df[column] = engines_df[column].astype('float')
except BaseException:
    print(column)
engines_df.info()

travel_charge
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141 entries, 0 to 140
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   name            141 non-null    object
 1   filename        141 non-null    object
 2   class           141 non-null    object
 3   page_id         141 non-null    int64 
 4   engine_id       141 non-null    int64 
 5   maker           138 non-null    object
 6   version         140 non-null    object
 7   boost_duration  141 non-null    int32 
 8   boost_thrust    141 non-null    object
 9   boost_attack    141 non-null    object
 10  boost_release   141 non-null    object
 11  travel_charge   141 non-null    object
 12  travel_thrust   141 non-null    object
 13  travel_attack   141 non-null    object
 14  travel_release  141 non-null    object
 15  thrust_forward  141 non-null    object
 16  thrust_reverse  140 non-null    object
dtypes: int32(1), int64(2), object(14)
memory

### Calculation of the number of S,M docks on ships <a class="anchor" id="id_3c"></a>

In [52]:
list_of_dockarea_files = [] # list of fiels with dock data 
for folder in dock_folders:
    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.endswith(".xml"):                
                #print(os.path.join(root, file))
                list_of_dockarea_files.append(os.path.join(root, file))
print(len(list_of_dockarea_files))

90


#### Exploring the data to get insights how this parameter is stored in the game files

In [53]:
with open(r'h:/Steam/X4/Unpacked/assets/units/size_xl/ship_arg_xl_carrier_01.xml', 'r') as user_file:
    file_contents = user_file.read()     
Sobject = BeautifulSoup(file_contents, features="xml")
tags = Sobject.findAll('connection', attrs = {'tags':"dockingbay "})
print(len(tags))

13


```<connection ref="con_dockarea_arg_s_ship_04">
        <macro ref="dockarea_arg_s_ship_04_macro" connection="Connection01" /> -- reference to the file with the number of S  docks
      </connection>
      <connection ref="con_dockarea_arg_xl_carrier_01">
        <macro ref="dockarea_arg_xl_carrier_01_macro" connection="Connection01" /> --reference to the file with the number of M docks
      </connection>
      <connection ref="con_dock_xs">
        <macro ref="dock_gen_xs_ship_01_macro" connection="Connection_component" />
      </connection>
      <connection ref="con_launchtube_arg_s_01_a"> -- count these to get the number of launchtubes
        <macro ref="launchtube_arg_s_01_macro" connection="Connection02" />
<connection ref="con_shipstorage01"> -- link to file with ship capacity
<macro ref="shipstorage_gen_m_eight_macro" connection="object" />
```

The example of the file with ship capacity:  
H:\Steam\X4\Unpacked\assets\props\SurfaceElements\macros\shipstorage_gen_s_01_macro.xml  
```
<macros>
  <macro name="shipstorage_gen_s_01_macro" class="dockingbay">
    <component ref="generic_dockingbay" />
    <properties>
      <identification name="{20104,79701}" description="{20104,79702}" />
      <dock capacity="40" external="0" storage="1" />
      <room walkable="0" />
      <docksize tags="dock_s" />
    </properties>
  </macro>
</macros>```

Get information about components of dock bays and ship storages from a test file:

In [54]:
file = r'H:\Steam\X4\Unpacked\assets\units\size_xl\macros\ship_arg_xl_carrier_01_a_macro.xml'
with open(file, 'r') as user_file:
        file_contents = user_file.read()             
        file_Sobject = BeautifulSoup(file_contents, features="xml")        

        test_data = {} # Test dict to store extracted information
        # get the value of class attribute     
        test_data['class'] = file_Sobject.select_one('component').get('class')
        # get info about dock components
        try:         
            dock_components = []
            tags = file_Sobject.findAll('connection', attrs = {'ref':re.compile('^\s*con_dockarea')})
            for x in tags:
                  dock_components.append(x['ref'].strip())
            test_data['dock_components'] = dock_components      
        except:
            test_data['dock_components'] = np.nan 
        # get info about ship storage bays
        try:         
            ship_storage_components = []
            tags = file_Sobject.findAll('connection', attrs = {'ref':re.compile('^\s*con_shipstorage')})
            for x in tags:
                  ship_storage_components.append(x['ref'].strip())
            test_data['ship_storage_components'] = ship_storage_components      
        except:
            test_data['ship_storage_components'] = np.nan 
test_data


{'class': None,
 'dock_components': ['con_dockarea_arg_s_ship_04',
  'con_dockarea_arg_xl_carrier_01'],
 'ship_storage_components': ['con_shipstorage01',
  'con_shipstorage02',
  'con_shipstorage03']}

Calculate the number of docking bays from a file with dock data:

In [55]:
file = r'h:\Steam\X4\Unpacked\assets\structures\dock\macros\dockarea_par_l_destroyer_01_macro.xml'
with open(file, 'r') as user_file:
        file_contents = user_file.read()             
        file_Sobject = BeautifulSoup(file_contents, features="xml")     

        file_data = {} # create dict for storeing data
        # Get filename, class and component       
        file_data['filename'] = file
        try:
            file_data['component'] =  file_Sobject.select_one('component').get('ref').strip()
        except:
            file_data['component'] =  np.nan
        #get name of the component
        try:
            file_data['name'] =  file_Sobject.select_one('macro').get('name').strip()
        except:
            file_data['name'] =  np.nan

        try:
            file_data['class'] =  file_Sobject.select_one('macro').get('class').strip()
        except:
            file_data['class'] =  np.nan
        # get text IDs
        try:
            t = file_Sobject.select_one('identification').get('name')
            file_data['name_page_id'] = re.findall('\d+',t)[0]
            file_data['name_id'] = re.findall('\d+',t)[1]
        except:
            file_data['name_page_id'] = np.nan
            file_data['name_id'] = np.nan
        try:                        
            #Calculate number of docks
            tags_xls = file_Sobject.findAll('connection', attrs = {'ref':re.compile('\s*(dockingbay)\s*')})
            #file_data['num_docking_bays'] = len(tags_xls)
            docks_dict = {}
            for tag in tags_xls:                                 
                dock = tag.select_one('macro').get('ref')                
                grade =  calculate_dock_class(dock)
                
                if grade not in docks_dict:
                    docks_dict[grade] = 1
                else:
                    docks_dict[grade] = docks_dict[grade] +1
            file_data['num_docks'] = docks_dict
        except:
            file_data['num_docks'] = np.nan   
file_data

{'filename': 'h:\\Steam\\X4\\Unpacked\\assets\\structures\\dock\\macros\\dockarea_par_l_destroyer_01_macro.xml',
 'component': 'dockarea_par_l_destroyer_01',
 'name': 'dockarea_par_l_destroyer_01_macro',
 'class': 'dockarea',
 'name_page_id': '20104',
 'name_id': '70001',
 'num_docks': {'M': 1, 'S': 2}}

#### Creating dataframe with the data about docking bays:

In [56]:
def calculate_dock_class(data):
    """Function to calculate the class of ship bays and ship storages"""
    try:
        if re.match('\w+_(s)_\w+', data):
            return "S"
        if re.match('\w+_(m)_\w+', data):
            return "M"
        if re.match('\w+_(l)_\w+', data):
            return "L"
        if re.match('\w+_(xl)_\w+', data):
            return "XL"
        if re.match('\w+_(xs)_\w+', data):
            return "XS"
    except:
            return np.nan
        
ship_docks_dict = {}
for file in list_of_dockarea_files:
    with open(file, 'r') as user_file:
        file_contents = user_file.read()            
        file_Sobject = BeautifulSoup(file_contents, features="xml")      

        file_data = {} # create dict for storeing data
        # Get name, class and component 
        try:
            file_data['name'] =  file_Sobject.select_one('macro').get('name').strip()
        except:
            file_data['name'] =  np.nan
        file_data['filename'] = file      
        try:
            component =  file_Sobject.select_one('component').get('ref').strip()
        except:
            file_data['component'] =  np.nan
       
        try:
            file_data['class'] =  file_Sobject.select_one('macro').get('class').strip()
        except:
            file_data['class'] =  np.nan

        # get text IDs
        try:
            t = file_Sobject.select_one('identification').get('name')
            file_data['name_page_id'] = re.findall('\d+',t)[0]
            file_data['name_id'] = re.findall('\d+',t)[1]           
        except:
            file_data['name_page_id'] = np.nan
            file_data['name_id'] = np.nan
     
        try:                        
            #Calculate number of docks
            tags_xls = file_Sobject.findAll('connection', attrs = {'ref':re.compile('\s*(dockingbay)\s*')})
            #file_data['num_docking_bays'] = len(tags_xls)
            docks_dict = {}
            for tag in tags_xls:                                 
                dock = tag.select_one('macro').get('ref')                
                grade =  calculate_dock_class(dock)
                
                if grade not in docks_dict:
                    docks_dict[grade] = 1
                else:
                    docks_dict[grade] = docks_dict[grade] +1
            file_data['num_docking_bays'] = docks_dict
        except:            
                file_data['num_docking_bays'] = np.nan 

        ship_docks_dict[component] = file_data
ship_docks_df = pd.DataFrame.from_dict(ship_docks_dict,orient = 'index').reset_index().rename(columns = {'index':'component'})
display(ship_docks_df.head())


,component,name,filename,class,name_page_id,name_id,num_docking_bays
0,dockarea_arg_m_02_tradestation_01,dockarea_arg_m_02_tradestation_01_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockarea,20104,73301,{'M': 8}
1,dockarea_arg_m_ship_01_hightech,dockarea_arg_m_ship_01_hightech_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockarea,20104,70001,{'M': 1}
2,dockarea_arg_m_ship_01,dockarea_arg_m_ship_01_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockarea,20104,70001,{'M': 1}
3,dockarea_arg_m_ship_02,dockarea_arg_m_ship_02_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockarea,20104,70001,{'M': 2}
4,dockarea_arg_m_ship_03,dockarea_arg_m_ship_03_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockarea,20104,70001,"{'M': 1, 'S': 3}"


Adding text data according to name_id from description file:

In [57]:
display(ship_docks_df['name_page_id'].unique())

array(['20104', nan], dtype=object)

As I have only one valid ```name_page_id``` I can iterate only through ```name_id``` to find text data:

In [58]:
display(ship_docks_df['name_page_id'].unique())
for x in ship_docks_df['name_id'].unique().tolist():
    try:
        ex = descriptions_Sobject.find('page', attrs = {'id':20104}).find('t',attrs = {'id':x})
        name = re.search('([A-Za-z0-9 -]+)',ex.text).group()
        ship_docks_df.loc[ship_docks_df['name_id']==x,'text']= name
    except:
        ship_docks_df.loc[ship_docks_df['name_id']==x,'text']= np.nan
display(ship_docks_df.head())

array(['20104', nan], dtype=object)

,component,name,filename,class,name_page_id,name_id,num_docking_bays,text
0,dockarea_arg_m_02_tradestation_01,dockarea_arg_m_02_tradestation_01_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockarea,20104,73301,{'M': 8},8M Luxury Dock Area
1,dockarea_arg_m_ship_01_hightech,dockarea_arg_m_ship_01_hightech_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockarea,20104,70001,{'M': 1},Dock Area
2,dockarea_arg_m_ship_01,dockarea_arg_m_ship_01_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockarea,20104,70001,{'M': 1},Dock Area
3,dockarea_arg_m_ship_02,dockarea_arg_m_ship_02_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockarea,20104,70001,{'M': 2},Dock Area
4,dockarea_arg_m_ship_03,dockarea_arg_m_ship_03_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockarea,20104,70001,"{'M': 1, 'S': 3}",Dock Area


Check for missing data:

In [59]:
missing_values_tab(ship_docks_df)

Dataframe contains 8 columns and 87 strings.

It has  3 columns with missed values


,missed_values,%_of_all,data_type
name_page_id,3,3.45,object
name_id,3,3.45,object
text,3,3.45,object


In [60]:
ship_docks_df[ship_docks_df.isna().any(axis = 1)].head()

,component,name,filename,class,name_page_id,name_id,num_docking_bays,text
25,dockarea_gen_m_inv_01,dockarea_gen_m_inv_01_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockarea,NaN,NaN,{'M': 1},NaN
26,dockarea_gen_s_inv_01,dockarea_gen_s_inv_01_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockarea,NaN,NaN,{'S': 1},NaN
46,dockingbay_gen_xl_inv,dockingbay_xen_lxl_inv_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockingbay,NaN,NaN,{},NaN


#### Creating dataframe with the data about ship capacity of docking bays:

In [61]:
ship_dock_capacity_dict = {} # dict to store data about dockbay capacity on ships
for file in list_of_shipstorage_files:
    with open(file, 'r') as user_file:
        file_contents = user_file.read()             
        file_Sobject = BeautifulSoup(file_contents, features="xml")      

        file_data = {} # create dict for storeing data
        # Get filename, class and component        
        try:
            file_data['component'] =  file_Sobject.select_one('component').get('ref').strip()
        except:
            file_data['component'] =  np.nan
        #get name
        try:
            file_data['name'] =  file_Sobject.select_one('macro').get('name').strip()
        except:
            file_data['name'] =  np.nan
        try:
            file_data['class'] =  file_Sobject.select_one('macro').get('class').strip()
        except:
            file_data['class'] =  np.nan
        # get text IDs
        try:
            t = file_Sobject.select_one('identification').get('name')
            file_data['name_page_id'] = re.findall('\d+',t)[0]
            file_data['name_id'] = re.findall('\d+',t)[1]           
        except:
            file_data['name_page_id'] = np.nan
            file_data['name_id'] = np.nan
        try:                        
            file_data['capacity'] =  int(file_Sobject.select_one('dock').get('capacity'))
        except:
            file_data['capacity'] = np.nan
        # find type of dock
        try:
            file_data['ship_storage_type'] = file_Sobject.select_one('docksize').get('tags').split('_')[1].upper()
        except:
            file_data['ship_storage_type'] = np.nan
        ship_dock_capacity_dict[file] = file_data
ship_docks_capacity_df = pd.DataFrame.from_dict(ship_dock_capacity_dict,orient = 'index').reset_index().rename(columns = {'index':'filename'})
for x in ship_docks_capacity_df['name_id'].unique().tolist():
    try:
        ex = descriptions_Sobject.find('page', attrs = {'id':20104}).find('t',attrs = {'id':x})
        name = re.search('([A-Za-z0-9 -]+)',ex.text).group()
        ship_docks_capacity_df.loc[ship_docks_capacity_df['name_id']==x,'text']= name
    except:
        ship_docks_capacity_df.loc[ship_docks_capacity_df['name_id']==x,'text']= np.nan
display(ship_docks_capacity_df.head())

,filename,component,name,class,name_page_id,name_id,capacity,ship_storage_type,text
0,h:\Steam\X4\Unpacked\assets\props\SurfaceEleme...,generic_dockingbay,shipstorage_gen_m_01_macro,dockingbay,20104,79701,10,M,Docking Bay
1,h:\Steam\X4\Unpacked\assets\props\SurfaceEleme...,generic_dockingbay,shipstorage_gen_m_02_macro,dockingbay,20104,79701,30,M,Docking Bay
2,h:\Steam\X4\Unpacked\assets\props\SurfaceEleme...,generic_dockingbay,shipstorage_gen_m_eight_macro,dockingbay,20104,79701,8,M,Docking Bay
3,h:\Steam\X4\Unpacked\assets\props\SurfaceEleme...,generic_dockingbay,shipstorage_gen_m_four_macro,dockingbay,20104,79701,4,M,Docking Bay
4,h:\Steam\X4\Unpacked\assets\props\SurfaceEleme...,generic_dockingbay,shipstorage_gen_m_two_macro,dockingbay,20104,79701,2,M,Docking Bay


In [62]:
ship_docks_capacity_df['ship_storage_type'].unique()

array(['M', 'S', 'XS'], dtype=object)

#### Creating dataframe with docking components used by ships

In [63]:
temp = [] # List of dicts containing info about components of ship bays and ship storages
error_files = []
for file in list_of_ships:
    if not file_validation(file):
            error_files.append(file) 
    else:             
        ship_data = {}
        with open(file, 'r') as user_file:
          file_contents = user_file.read()             
          soup = BeautifulSoup(file_contents, features="xml")           
          ship_data['component'] = soup.select_one('component').get('ref')
          ship_data['filename'] = soup.select_one('macro').get('name')
          ship_data['ship_class'] = soup.select_one('macro').get('class')          
            
                # get info about dock components
          dock_components = []
          try:           
            tags = soup.findAll('macro', attrs = {'ref':re.compile('^\s*dockarea')})
            for x in tags:
                  dock_components.append(x['ref'].strip())
            ship_data['dock_components'] = dock_components      
          except:
            ship_data['dock_components'] = np.nan 
        # get info about ship storage bays
          try:         
            ship_storage_components = []
            tags = soup.findAll('macro', attrs = {'ref':re.compile('^\s*shipstorage')})
            for x in tags:
                  ship_storage_components.append(x['ref'].strip())
            ship_data['ship_storage_components'] = ship_storage_components      
          except:
            ship_data['ship_storage_components'] = np.nan 
          temp.append(ship_data)
## Creating table about links to dock and storage components used by ships    
ship_dock_componentes_df = pd.DataFrame.from_dict(temp)
# create columns with lengths of lists
ship_dock_componentes_df['dock_components_num'] = ship_dock_componentes_df['dock_components'].str.len()
ship_dock_componentes_df['ship_storage_comp_num'] = ship_dock_componentes_df['ship_storage_components'].str.len()
display(ship_dock_componentes_df.head())

,component,filename,ship_class,dock_components,ship_storage_components,dock_components_num,ship_storage_comp_num
0,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_a_macro,ship_l,[dockarea_arg_s_ship_03_macro],"[shipstorage_gen_s_01_macro, shipstorage_gen_x...",1,2
1,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_b_macro,ship_l,[dockarea_arg_s_ship_03_macro],"[shipstorage_gen_s_01_macro, shipstorage_gen_x...",1,2
2,ship_arg_l_destroyer_02,ship_arg_l_destroyer_02_a_macro,ship_l,[dockarea_arg_s_ship_03_macro],"[shipstorage_gen_s_eight_macro, shipstorage_ge...",1,2
3,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_a_macro,ship_l,"[dockarea_arg_s_ship_01_macro, dockarea_arg_s_...","[shipstorage_gen_s_01_macro, shipstorage_gen_x...",2,2
4,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_b_macro,ship_l,"[dockarea_arg_s_ship_01_macro, dockarea_arg_s_...","[shipstorage_gen_s_01_macro, shipstorage_gen_x...",2,2


Unfolding columns containing lists of dcomponents into separate columns:

In [64]:
print(ship_dock_componentes_df['dock_components_num'].max())
print(ship_dock_componentes_df['ship_storage_comp_num'].max())

4
3


In [65]:
ship_dock_componentes_df[
    ['storage_comp_1','storage_comp_2', 'storage_comp_3']
    ] = pd.DataFrame(ship_dock_componentes_df['ship_storage_components'].tolist(), 
                                                                       index = ship_dock_componentes_df.index)

In [66]:
ship_dock_componentes_df[
    ['dock_comp_1','dock_comp_2','dock_comp_3', 'dock_comp_4']
    ] = pd.DataFrame(ship_dock_componentes_df['dock_components'].tolist(), 
                                                                       index = ship_dock_componentes_df.index)


In [67]:
display(ship_dock_componentes_df.head())

,component,filename,ship_class,dock_components,ship_storage_components,dock_components_num,ship_storage_comp_num,storage_comp_1,storage_comp_2,storage_comp_3,dock_comp_1,dock_comp_2,dock_comp_3,dock_comp_4
0,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_a_macro,ship_l,[dockarea_arg_s_ship_03_macro],"[shipstorage_gen_s_01_macro, shipstorage_gen_x...",1,2,shipstorage_gen_s_01_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_03_macro,None,None,None
1,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_b_macro,ship_l,[dockarea_arg_s_ship_03_macro],"[shipstorage_gen_s_01_macro, shipstorage_gen_x...",1,2,shipstorage_gen_s_01_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_03_macro,None,None,None
2,ship_arg_l_destroyer_02,ship_arg_l_destroyer_02_a_macro,ship_l,[dockarea_arg_s_ship_03_macro],"[shipstorage_gen_s_eight_macro, shipstorage_ge...",1,2,shipstorage_gen_s_eight_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_03_macro,None,None,None
3,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_a_macro,ship_l,"[dockarea_arg_s_ship_01_macro, dockarea_arg_s_...","[shipstorage_gen_s_01_macro, shipstorage_gen_x...",2,2,shipstorage_gen_s_01_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_01_macro,dockarea_arg_s_ship_02_macro,None,None
4,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_b_macro,ship_l,"[dockarea_arg_s_ship_01_macro, dockarea_arg_s_...","[shipstorage_gen_s_01_macro, shipstorage_gen_x...",2,2,shipstorage_gen_s_01_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_01_macro,dockarea_arg_s_ship_02_macro,None,None


In [68]:
ex = descriptions_Sobject.find('page', attrs = {'id':20104}).find('t',attrs = {'id':79701})
name = re.search('([A-Za-z0-9 -]+)',ex.text).group()
name

'Docking Bay'

#### Merging dataframes

In [69]:
display(ship_docks_capacity_df.head(2))
display(ship_docks_df.head(2))

,filename,component,name,class,name_page_id,name_id,capacity,ship_storage_type,text
0,h:\Steam\X4\Unpacked\assets\props\SurfaceEleme...,generic_dockingbay,shipstorage_gen_m_01_macro,dockingbay,20104,79701,10,M,Docking Bay
1,h:\Steam\X4\Unpacked\assets\props\SurfaceEleme...,generic_dockingbay,shipstorage_gen_m_02_macro,dockingbay,20104,79701,30,M,Docking Bay


,component,name,filename,class,name_page_id,name_id,num_docking_bays,text
0,dockarea_arg_m_02_tradestation_01,dockarea_arg_m_02_tradestation_01_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockarea,20104,73301,{'M': 8},8M Luxury Dock Area
1,dockarea_arg_m_ship_01_hightech,dockarea_arg_m_ship_01_hightech_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockarea,20104,70001,{'M': 1},Dock Area


Merge ```ship_dock_componentes_df``` with ```ship_docks_capacity_df``` by 3 columns:

In [70]:
ship_dock_componentes_df = (ship_dock_componentes_df.merge(ship_docks_capacity_df[['name','capacity','text', 'ship_storage_type']],
                                                          left_on = ['storage_comp_1'],
                                                          right_on = ['name'],
                                                          how = 'left',
                                                          suffixes=(None,'_y')).drop(columns = 'name', axis = 1)
                                                          .rename(columns = {'capacity':'ship_capacity_1',
                                                                             'text':'text_1','ship_storage_type':'ship_storage_type_1'}))
ship_dock_componentes_df = (ship_dock_componentes_df.merge(ship_docks_capacity_df[['name','capacity','text','ship_storage_type']],
                                                          left_on = ['storage_comp_2'],
                                                          right_on = ['name'],
                                                          how = 'left',
                                                          suffixes=(None,'_y')).drop(columns = 'name', axis = 1)
                                                          .rename(columns = {'capacity':'ship_capacity_2',
                                                                             'text':'text_2','ship_storage_type':'ship_storage_type_2'}))
ship_dock_componentes_df = (ship_dock_componentes_df.merge(ship_docks_capacity_df[['name','capacity','text','ship_storage_type']],
                                                          left_on = ['storage_comp_3'],
                                                          right_on = ['name'],
                                                          how = 'left',
                                                          suffixes=(None,'_y')).drop(columns = 'name', axis = 1)
                                                          .rename(columns = {'capacity':'ship_capacity_3',
                                                                             'text':'text_3','ship_storage_type':'ship_storage_type_3'}))

Merge ```ship_dock_componentes_df``` with ```ship_docks_df``` by 4 columns:

In [71]:
display(ship_docks_df.head(1))
display(ship_dock_componentes_df.head(1))

,component,name,filename,class,name_page_id,name_id,num_docking_bays,text
0,dockarea_arg_m_02_tradestation_01,dockarea_arg_m_02_tradestation_01_macro,h:\Steam\X4\Unpacked\assets\structures\dock\ma...,dockarea,20104,73301,{'M': 8},8M Luxury Dock Area


,component,filename,ship_class,dock_components,ship_storage_components,dock_components_num,ship_storage_comp_num,storage_comp_1,storage_comp_2,storage_comp_3,dock_comp_1,dock_comp_2,dock_comp_3,dock_comp_4,ship_capacity_1,text_1,ship_storage_type_1,ship_capacity_2,text_2,ship_storage_type_2,ship_capacity_3,text_3,ship_storage_type_3
0,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_a_macro,ship_l,[dockarea_arg_s_ship_03_macro],"[shipstorage_gen_s_01_macro, shipstorage_gen_x...",1,2,shipstorage_gen_s_01_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_03_macro,None,None,None,40.0,Docking Bay,S,10.0,Docking Bay,XS,NaN,NaN,NaN


In [72]:
ship_dock_componentes_df = (ship_dock_componentes_df.merge(ship_docks_df[['name', 'num_docking_bays']],
                                                          left_on = ['dock_comp_1'],
                                                          right_on = ['name'],
                                                          how = 'left',
                                                          suffixes=(None,'_y')).drop(columns = 'name', axis = 1)
                                                          .rename(columns = {'num_docking_bays':'num_docking_bays_1'}))
ship_dock_componentes_df = (ship_dock_componentes_df.merge(ship_docks_df[['name', 'num_docking_bays']],
                                                          left_on = ['dock_comp_2'],
                                                          right_on = ['name'],
                                                          how = 'left',
                                                          suffixes=(None,'_y')).drop(columns = 'name', axis = 1)
                                                          .rename(columns = {'num_docking_bays':'num_docking_bays_2'}))
ship_dock_componentes_df = (ship_dock_componentes_df.merge(ship_docks_df[['name', 'num_docking_bays']],
                                                          left_on = ['dock_comp_3'],
                                                          right_on = ['name'],
                                                          how = 'left',
                                                          suffixes=(None,'_y')).drop(columns = 'name', axis = 1)
                                                          .rename(columns = {'num_docking_bays':'num_docking_bays_3'}))
ship_dock_componentes_df = (ship_dock_componentes_df.merge(ship_docks_df[['name', 'num_docking_bays']],
                                                          left_on = ['dock_comp_4'],
                                                          right_on = ['name'],
                                                          how = 'left',
                                                          suffixes=(None,'_y')).drop(columns = 'name', axis = 1)
                                                          .rename(columns = {'num_docking_bays':'num_docking_bays_4'}))

In [73]:
ship_dock_componentes_df.head()

,component,filename,ship_class,dock_components,ship_storage_components,dock_components_num,ship_storage_comp_num,storage_comp_1,storage_comp_2,storage_comp_3,dock_comp_1,dock_comp_2,dock_comp_3,dock_comp_4,ship_capacity_1,text_1,ship_storage_type_1,ship_capacity_2,text_2,ship_storage_type_2,ship_capacity_3,text_3,ship_storage_type_3,num_docking_bays_1,num_docking_bays_2,num_docking_bays_3,num_docking_bays_4
0,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_a_macro,ship_l,[dockarea_arg_s_ship_03_macro],"[shipstorage_gen_s_01_macro, shipstorage_gen_x...",1,2,shipstorage_gen_s_01_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_03_macro,None,None,None,40.0,Docking Bay,S,10.0,Docking Bay,XS,NaN,NaN,NaN,{'S': 4},NaN,NaN,NaN
1,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_b_macro,ship_l,[dockarea_arg_s_ship_03_macro],"[shipstorage_gen_s_01_macro, shipstorage_gen_x...",1,2,shipstorage_gen_s_01_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_03_macro,None,None,None,40.0,Docking Bay,S,10.0,Docking Bay,XS,NaN,NaN,NaN,{'S': 4},NaN,NaN,NaN
2,ship_arg_l_destroyer_02,ship_arg_l_destroyer_02_a_macro,ship_l,[dockarea_arg_s_ship_03_macro],"[shipstorage_gen_s_eight_macro, shipstorage_ge...",1,2,shipstorage_gen_s_eight_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_03_macro,None,None,None,8.0,Docking Bay,S,10.0,Docking Bay,XS,NaN,NaN,NaN,{'S': 4},NaN,NaN,NaN
3,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_a_macro,ship_l,"[dockarea_arg_s_ship_01_macro, dockarea_arg_s_...","[shipstorage_gen_s_01_macro, shipstorage_gen_x...",2,2,shipstorage_gen_s_01_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_01_macro,dockarea_arg_s_ship_02_macro,None,None,40.0,Docking Bay,S,10.0,Docking Bay,XS,NaN,NaN,NaN,{'S': 1},{'S': 2},NaN,NaN
4,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_b_macro,ship_l,"[dockarea_arg_s_ship_01_macro, dockarea_arg_s_...","[shipstorage_gen_s_01_macro, shipstorage_gen_x...",2,2,shipstorage_gen_s_01_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_01_macro,dockarea_arg_s_ship_02_macro,None,None,40.0,Docking Bay,S,10.0,Docking Bay,XS,NaN,NaN,NaN,{'S': 1},{'S': 2},NaN,NaN


Create separate dataframes from columns containing dictionaries then concatenate them into the main table:

In [74]:
num_bays_1 = ship_dock_componentes_df['num_docking_bays_1'].apply(pd.Series).drop(0,axis = 1).rename(columns = {'S': 'S_docks_1', 'M':'M_docks_1'})
num_bays_2 = ship_dock_componentes_df['num_docking_bays_2'].apply(pd.Series).drop(0,axis = 1).rename(columns = {'S': 'S_docks_2', 'M':'M_docks_2'})
num_bays_3 = ship_dock_componentes_df['num_docking_bays_3'].apply(pd.Series).drop(0,axis = 1).rename(columns = {'S': 'S_docks_3', 'M':'M_docks_3'})
num_bays_4 = ship_dock_componentes_df['num_docking_bays_4'].apply(pd.Series).drop(0,axis = 1).rename(columns = {'S': 'S_docks_4', 'M':'M_docks_4'})
display(num_bays_2)

,S_docks_2,M_docks_2
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,2.0,NaN
4,2.0,NaN
...,...,...
345,NaN,NaN
346,NaN,NaN
347,NaN,NaN
348,NaN,NaN


In [75]:
ship_dock_componentes_df = pd.concat([ship_dock_componentes_df.drop('num_docking_bays_1',axis = 1), num_bays_1],axis = 1)
ship_dock_componentes_df = pd.concat([ship_dock_componentes_df.drop('num_docking_bays_2',axis = 1), num_bays_2],axis = 1)
ship_dock_componentes_df = pd.concat([ship_dock_componentes_df.drop('num_docking_bays_3',axis = 1), num_bays_3],axis = 1)
ship_dock_componentes_df = pd.concat([ship_dock_componentes_df.drop('num_docking_bays_4',axis = 1), num_bays_4],axis = 1)


In [76]:
# calculate total number of S/M docks
s_cols = ['S_docks_1', 'S_docks_2',  'S_docks_3', 'S_docks_4']
m_cols = ['M_docks_1', 'M_docks_2', 'M_docks_3']
ship_dock_componentes_df['num_S_docks'] = ship_dock_componentes_df[s_cols].sum(axis = 1)
ship_dock_componentes_df['num_M_docks'] = ship_dock_componentes_df[m_cols].sum(axis = 1)
# Convert values to int
ship_dock_componentes_df['num_S_docks'] = ship_dock_componentes_df['num_S_docks'].astype('int')
ship_dock_componentes_df['num_M_docks'] = ship_dock_componentes_df['num_M_docks'].astype('int')
ship_dock_componentes_df['num_S_docks'].unique()

array([ 4,  3,  1,  2,  0,  8,  6, 16, 18, 21])

Calculating the class of bays and ship storages:

In [77]:
ship_dock_componentes_df['storage_comp_1_class'] = ship_dock_componentes_df['storage_comp_1'].apply(calculate_dock_class)
ship_dock_componentes_df['storage_comp_2_class'] = ship_dock_componentes_df['storage_comp_2'].apply(calculate_dock_class)
ship_dock_componentes_df['storage_comp_3_class'] = ship_dock_componentes_df['storage_comp_3'].apply(calculate_dock_class)


Calculate the total number of docks:

In [78]:
ship_dock_componentes_df.head()

,component,filename,ship_class,dock_components,ship_storage_components,dock_components_num,ship_storage_comp_num,storage_comp_1,storage_comp_2,storage_comp_3,dock_comp_1,dock_comp_2,dock_comp_3,dock_comp_4,ship_capacity_1,text_1,ship_storage_type_1,ship_capacity_2,text_2,ship_storage_type_2,ship_capacity_3,text_3,ship_storage_type_3,S_docks_1,M_docks_1,S_docks_2,M_docks_2,M_docks_3,S_docks_3,S_docks_4,num_S_docks,num_M_docks,storage_comp_1_class,storage_comp_2_class,storage_comp_3_class
0,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_a_macro,ship_l,[dockarea_arg_s_ship_03_macro],"[shipstorage_gen_s_01_macro, shipstorage_gen_x...",1,2,shipstorage_gen_s_01_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_03_macro,None,None,None,40.0,Docking Bay,S,10.0,Docking Bay,XS,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,4,0,S,XS,NaN
1,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_b_macro,ship_l,[dockarea_arg_s_ship_03_macro],"[shipstorage_gen_s_01_macro, shipstorage_gen_x...",1,2,shipstorage_gen_s_01_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_03_macro,None,None,None,40.0,Docking Bay,S,10.0,Docking Bay,XS,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,4,0,S,XS,NaN
2,ship_arg_l_destroyer_02,ship_arg_l_destroyer_02_a_macro,ship_l,[dockarea_arg_s_ship_03_macro],"[shipstorage_gen_s_eight_macro, shipstorage_ge...",1,2,shipstorage_gen_s_eight_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_03_macro,None,None,None,8.0,Docking Bay,S,10.0,Docking Bay,XS,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,4,0,S,XS,NaN
3,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_a_macro,ship_l,"[dockarea_arg_s_ship_01_macro, dockarea_arg_s_...","[shipstorage_gen_s_01_macro, shipstorage_gen_x...",2,2,shipstorage_gen_s_01_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_01_macro,dockarea_arg_s_ship_02_macro,None,None,40.0,Docking Bay,S,10.0,Docking Bay,XS,NaN,NaN,NaN,1.0,NaN,2.0,NaN,NaN,NaN,NaN,3,0,S,XS,NaN
4,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_b_macro,ship_l,"[dockarea_arg_s_ship_01_macro, dockarea_arg_s_...","[shipstorage_gen_s_01_macro, shipstorage_gen_x...",2,2,shipstorage_gen_s_01_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_01_macro,dockarea_arg_s_ship_02_macro,None,None,40.0,Docking Bay,S,10.0,Docking Bay,XS,NaN,NaN,NaN,1.0,NaN,2.0,NaN,NaN,NaN,NaN,3,0,S,XS,NaN


Calculate the total ship capacity:

In [79]:
ship_dock_componentes_df.head(1)

,component,filename,ship_class,dock_components,ship_storage_components,dock_components_num,ship_storage_comp_num,storage_comp_1,storage_comp_2,storage_comp_3,dock_comp_1,dock_comp_2,dock_comp_3,dock_comp_4,ship_capacity_1,text_1,ship_storage_type_1,ship_capacity_2,text_2,ship_storage_type_2,ship_capacity_3,text_3,ship_storage_type_3,S_docks_1,M_docks_1,S_docks_2,M_docks_2,M_docks_3,S_docks_3,S_docks_4,num_S_docks,num_M_docks,storage_comp_1_class,storage_comp_2_class,storage_comp_3_class
0,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_a_macro,ship_l,[dockarea_arg_s_ship_03_macro],"[shipstorage_gen_s_01_macro, shipstorage_gen_x...",1,2,shipstorage_gen_s_01_macro,shipstorage_gen_xs_01_macro,None,dockarea_arg_s_ship_03_macro,None,None,None,40.0,Docking Bay,S,10.0,Docking Bay,XS,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,4,0,S,XS,NaN


In [80]:
# capacity S ships
ship_dock_componentes_df['capacity_S_ships'] = np.where(ship_dock_componentes_df['ship_storage_type_1']=="S",
                                                   ship_dock_componentes_df['ship_capacity_1'],0)
ship_dock_componentes_df['capacity_S_ships'] = np.where(ship_dock_componentes_df['ship_storage_type_2']=="S",
                                                   ship_dock_componentes_df['capacity_S_ships'] 
                                                   + ship_dock_componentes_df['ship_capacity_2'], ship_dock_componentes_df['capacity_S_ships'])
ship_dock_componentes_df['capacity_S_ships'] = np.where(ship_dock_componentes_df['ship_storage_type_3']=="S",
                                                   ship_dock_componentes_df['ship_capacity_3'] 
                                                   + ship_dock_componentes_df['capacity_S_ships'], ship_dock_componentes_df['capacity_S_ships'])

#capacity M ships
ship_dock_componentes_df['capacity_M_ships'] = np.where(ship_dock_componentes_df['ship_storage_type_1']=="M",
                                                   ship_dock_componentes_df['ship_capacity_1'],0)
ship_dock_componentes_df['capacity_M_ships'] = np.where(ship_dock_componentes_df['ship_storage_type_2']=="M",
                                                   ship_dock_componentes_df['capacity_M_ships'] 
                                                   + ship_dock_componentes_df['ship_capacity_2'], ship_dock_componentes_df['capacity_M_ships'])
ship_dock_componentes_df['capacity_M_ships'] = np.where(ship_dock_componentes_df['ship_storage_type_3']=="M",
                                                   ship_dock_componentes_df['ship_capacity_3'] 
                                                   + ship_dock_componentes_df['capacity_M_ships'], ship_dock_componentes_df['capacity_M_ships'])

#L ships capacity
ship_dock_componentes_df['capacity_L_ships'] = np.where(ship_dock_componentes_df['ship_storage_type_1']=="L",
                                                   ship_dock_componentes_df['ship_capacity_1'],0)
ship_dock_componentes_df['capacity_L_ships'] = np.where(ship_dock_componentes_df['ship_storage_type_2']=="L",
                                                   ship_dock_componentes_df['capacity_L_ships'] 
                                                   + ship_dock_componentes_df['ship_capacity_2'], ship_dock_componentes_df['capacity_L_ships'])
ship_dock_componentes_df['capacity_L_ships'] = np.where(ship_dock_componentes_df['ship_storage_type_3']=="L",
                                                   ship_dock_componentes_df['ship_capacity_3'] 
                                                   + ship_dock_componentes_df['capacity_L_ships'], ship_dock_componentes_df['capacity_L_ships'])
#XS capacity
ship_dock_componentes_df['capacity_XS_ships'] = np.where(ship_dock_componentes_df['ship_storage_type_1']=="XS",
                                                   ship_dock_componentes_df['ship_capacity_1'],0)
ship_dock_componentes_df['capacity_XS_ships'] = np.where(ship_dock_componentes_df['ship_storage_type_2']=="XS",
                                                   ship_dock_componentes_df['capacity_XS_ships'] 
                                                   + ship_dock_componentes_df['ship_capacity_2'], ship_dock_componentes_df['capacity_XS_ships'])
ship_dock_componentes_df['capacity_XS_ships'] = np.where(ship_dock_componentes_df['ship_storage_type_3']=="XS",
                                                   ship_dock_componentes_df['ship_capacity_3'] 
                                                   + ship_dock_componentes_df['capacity_XS_ships'], ship_dock_componentes_df['capacity_XS_ships'])

#### Merging ```ship_data_df``` and ```ship_dock_componentes_df```

In [81]:
cols_to_merge = ['filename', 'num_S_docks',	'num_M_docks',	'capacity_S_ships',	'capacity_M_ships',	'capacity_XS_ships']
ship_data_df = ship_data_df.merge(ship_dock_componentes_df[cols_to_merge],
                                   how = 'left', 
                                  on = 'filename',
                                  suffixes=(None,'_y'))

In [82]:
cols_to_int = ['num_S_docks',	'num_M_docks',	'capacity_S_ships',
                  	'capacity_M_ships',	'capacity_XS_ships']
ship_data_df[cols_to_int] = ship_data_df[cols_to_int].astype('int')

### Data corrections

### Final result

Increasing the crew count by 1 for all ships where it is equl to 0:

In [83]:
ship_data_df['people_capacity'] = np.where(ship_data_df['people_capacity']!=0, ship_data_df['people_capacity']+1,ship_data_df['people_capacity'])

Drop the columns```file, capacity_XS_ships```:

In [84]:
ship_data_df = ship_data_df.drop(['file','capacity_XS_ships'], axis = 1)

In [85]:
ship_data_df.head()

,component,filename,ship_class,name_page_id,name_id,maker,storage_missile,storage_unit,hull,purpose,people_capacity,ship_mass,inertia_pitch,inertia_yaw,inertia_roll,drag_forward,drag_reverse,drag_horizontal,drag_vertical,drag_pitch,drag_yaw,drag_roll,ship_type,storage_file,cargo_volume,storage_type,name,primary_shield_slots,turret_shield_slots,l_turret_slots,m_turret_slots,primary_weapon_slots,engines_slots,num_S_docks,num_M_docks,capacity_S_ships,capacity_M_ships
0,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_a_macro,ship_l,20101,11002,argon,160,10,93000,fight,45,196.016,96.271,96.271,77.016,99.004,396.016,73.005,73.005,106.203,106.203,106.203,destroyer,storage_arg_l_destroyer_01_a_macro,2300,container,Behemoth Vanguard,3,9,2,8,2,3,4,0,40,0
1,ship_arg_l_destroyer_01,ship_arg_l_destroyer_01_b_macro,ship_l,20101,11003,argon,160,10,111000,fight,37,235.220,103.378,103.378,82.702,108.805,435.220,87.605,87.605,114.044,114.044,114.044,destroyer,storage_arg_l_destroyer_01_b_macro,2760,container,Behemoth Sentinel,3,9,2,8,2,3,4,0,40,0
2,ship_arg_l_destroyer_02,ship_arg_l_destroyer_02_a_macro,ship_l,20101,11004,argon,160,10,102000,fight,49,260.472,143.455,143.455,114.764,80.583,460.472,43.260,43.260,119.094,119.094,119.094,destroyer,storage_arg_l_destroyer_02_a_macro,3100,container,Behemoth E,3,9,2,8,2,3,4,0,8,0
3,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_a_macro,ship_l,20101,11104,argon,30,10,26000,mine,47,205.270,133.749,133.749,106.999,56.738,324.216,126.666,126.666,140.897,140.897,140.897,largeminer,storage_arg_l_miner_liquid_01_a_macro,42000,liquid,Magnetar Gas Vanguard,2,7,0,6,0,2,3,0,40,0
4,ship_arg_l_miner_liquid_01,ship_arg_l_miner_liquid_01_b_macro,ship_l,20101,11105,argon,30,10,32000,mine,39,246.324,147.778,147.778,118.223,62.485,357.059,151.999,151.999,155.677,155.677,155.677,largeminer,storage_arg_l_miner_liquid_01_b_macro,50400,liquid,Magnetar Gas Sentinel,2,7,0,6,0,2,3,0,40,0


## Saving tables <a class="anchor" id="id_4"></a>

Ships:

In [86]:
#Making a deep copy of the table
saveCopy_df = ship_data_df.copy(deep = True)
# Saving in csv 
saveCopy_df.to_csv(save_folder + r'\raw_X4v7data.csv', index=False)
#Saving in excel
saveCopy_df.to_excel(save_folder + r'\raw_X4v7data.xlsx',  sheet_name='ship_data',index=False)

Shields:

In [87]:
# making the copy of the table
shields_saveCopy_df = shields_df.copy(deep = True)
# csv
shields_saveCopy_df.to_csv(save_folder + r'\shields_X4v7data.csv', index=False)
# excel
shields_saveCopy_df.to_excel(save_folder + r'\shields_X4v7data.xlsx',  sheet_name='shield_data',index=False)

Engines:

In [88]:
# making the copy of the table
engines_saveCopy_df = engines_df.copy(deep = True)
# csv
engines_saveCopy_df.to_csv(save_folder + r'\engines_X4v7data.csv', index=False)
# excel
engines_saveCopy_df.to_excel(save_folder + r'\engines_X4v7data.xlsx',  sheet_name='engine_data',index=False)

_____